# 🛍️ ShopTalk — Phase 10: Voice input

Speech-to-text is added **in front of** the unchanged Phase 6–9 pipeline (parse → retrieve → rerank → generate).

| § | What | Plan item |
|---|---|---|
| 1 | Setup: GPU, Drive, pinned deps (+ `python-multipart`), backups of the files Phase 10 changes | — |
| 2 | Code: `src/asr/` (Whisper), `src/tts/` (optional voice), new API endpoints, microphone in the UI | 10.1–10.4 |
| 3 | Test set: **record 20 spoken queries** (a recorder page with a 🎤 button) + a synthetic set for all 63 queries | 10.5 |
| 4 | Compare Whisper sizes on word error rate (WER) and speed; pick the default | 10.1 |
| 5 | Start the API with speech-to-text loaded once at startup | 10.1–10.2 |
| 6 | Voice vs text through the API: WER, key words (brands, sizes, prices), parsed filters and P@5, measured separately | 10.5 |
| 7 | Latency: speech-to-text P50/P95/P99 added to the Phase 9 report | 10.5 |
| 8 | Optional text-to-speech check | 10.4 |
| 9 | UI with microphone and transcript (manual test) | 10.3 |
| 10 | Write-up `asr_report.md` · 11 Exit checklist | — |

**What changes in the repo** (backed up first in 1.5): `src/api/main.py`, `schemas.py`, `service.py`, `src/ui/app.py`,
`client.py`, `render.py`; new `src/asr/` and `src/tts/`. The text endpoints and their responses are unchanged, and the
API still works with `SHOPTALK_ASR=0` (voice off).

**New endpoints:** `POST /transcribe` (audio → query text), `POST /recommend/audio` (audio → same pipeline),
`POST /speak` (text → WAV, only when `SHOPTALK_TTS=1`). `/health` reports `asr` / `tts`.

**Spoken numbers.** Whisper sometimes writes prices as words ("under thirty dollars"). The rule parser does not read
those, and the product-type lexicon even maps "thirty" to HAT. So the transcript is normalised before searching
("under $30"); the UI shows the normalised text, and §6 measures both versions.

**Run order:** GPU (T4) runtime → run 1–3.1 → **record in 3.2** (5–10 min) → run the rest. Sections 3.2 and 9 need you;
everything else runs unattended (≈ 40–60 min).

## 1. Setup

### 📘 1.1 Check the GPU and mount Google Drive

**Purpose:** Confirm a CUDA GPU (Whisper and Qwen share it), then mount Drive, with fallbacks for the Colab sign-in bug seen in Phase 9.

**Expected Results:** The GPU name (`Tesla T4`) and `✅ Drive mounted`.

**Parameters:** none

In [ ]:
import os, shutil, subprocess
if shutil.which("nvidia-smi") is None:
    print("⚠️ No NVIDIA GPU on this runtime. Runtime ▸ Change runtime type ▸ T4 GPU, then re-run from here.")
else:
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip())

from google.colab import drive
MYDRIVE = "/content/drive/MyDrive"
if os.path.isdir(MYDRIVE):
    print("Drive already mounted — skipping mount")
else:
    errors = []
    try:
        drive.mount("/content/drive", force_remount=True)
    except Exception as e:
        errors.append(f"drive.mount: {e}")
        try:
            from google.colab import auth
            auth.authenticate_user()
            drive.mount("/content/drive", force_remount=True)
        except Exception as e2:
            errors.append(f"auth.authenticate_user + drive.mount: {e2}")
    if not os.path.isdir(MYDRIVE):
        print("❌ Drive mount failed:", *errors, sep="\n  ")
        print("Try: Files sidebar (📁) ▸ 'Mount Drive', then re-run this cell; or Firefox; or an incognito window.")
        raise RuntimeError("Google Drive not mounted")
print("✅ Drive mounted")

### 📘 1.2 Configuration (single config cell)

**Purpose:** All Phase 10 parameters in one place.

**Expected Results:** No output.

**Parameters:** paths · test set · Whisper candidates and selection rule · API · targets · latency · UI — see inline comments.

In [ ]:
# ===================== Phase 10 configuration — the only cell you should need to edit =====================
from pathlib import Path
import time

# --- Paths
DRIVE_PROJECT = Path("/content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk")
REPO_LINK     = Path("/content/ShopTalk")            # symlink (no spaces → safe for %%writefile)
RAG_CONFIG_REL = "src/rag/rag_config.json"
RESULTS_DIR   = REPO_LINK / "results" / "phase10"
PHASE9_DIR    = REPO_LINK / "results" / "phase9"
LOG_DIR       = Path("/content/phase10_logs")         # Phase 10 traffic never goes into the production logs/
LOCAL_CHROMA  = "/content/shoptalk_chroma"            # the API copies the Chroma index here (local disk)
RUN_ID        = time.strftime("%Y%m%d-%H%M")
SESSION_PREFIX = f"phase10-{RUN_ID}"                   # add "phase10-" to TEST_SESSION_PREFIXES when mining logs

# --- Test set
EVAL_PATH       = None        # None = find eval_queries_expanded.json (the Phase 9 set); or a repo-relative path
N_RECORD        = 20          # plan: 10–20 spoken queries
MIN_RECORDINGS  = 10
SPEAKER         = "me"        # default speaker id on the recorder page (friends can type their own)
RECORD_DIR      = RESULTS_DIR / "recordings"          # <speaker>/<qid>.wav, 16 kHz mono
SYN_DIR         = RESULTS_DIR / "synthetic"           # TTS versions of all eval queries
MAKE_SYNTHETIC  = True

# --- Speech-to-text models (Hugging Face ids) and selection rule
ASR_CANDIDATES  = ["openai/whisper-tiny.en", "openai/whisper-base.en", "openai/whisper-small.en",
                   "distil-whisper/distil-small.en", "openai/whisper-large-v3-turbo"]
ASR_MODEL       = None        # None = choose in §4: lowest WER within ASR_P95_BUDGET_MS, then fastest within tolerance
ASR_P95_BUDGET_MS = 500       # per-clip speech-to-text P95 allowed on the T4
ASR_WER_TOLERANCE = 0.02      # models this close to the best WER count as equally accurate → pick the fastest
APPLY_ASR_CHOICE  = True      # write the chosen model into rag_config.json["asr"] (backup kept)

# --- API
API_APP          = "src.api.main:app"
API_PORT         = 8000
USE_FAKE         = False      # True → SHOPTALK_FAKE=1 dry run (fake ASR/LLM; numbers meaningless)
ENABLE_TTS       = True       # server loads the read-aloud voice; the UI checkbox is still OFF by default
HEALTH_TIMEOUT_S = 900

# --- Targets (the plan sets none for voice; these are the defaults this notebook checks against)
WER_TARGET               = 0.15   # corpus WER on your recordings
FILTER_AGREEMENT_TARGET  = 0.85   # share of clips whose parsed filters equal the typed query's
PRECISION_DROP_TOLERANCE = 0.05   # voice P@5 may be at most this much below typed P@5
TEXT_LATENCY_TARGET_MS   = 5000   # Phase 7–9 target for the text path
VOICE_E2E_TARGET_MS      = 5500   # text target + speech-to-text budget

# --- Latency runs
N_ASR_LATENCY   = 100
N_VOICE_E2E     = 100
LATENCY_WARMUP  = 3
RUN_SYNTH_E2E   = True        # also send the 63 synthetic clips through /recommend/audio in §6

# --- UI / recorder
UI_PORT         = 7860
REC_PORT        = 7861
USE_CLOUDFLARED = True        # https tunnel (the browser only allows the microphone on https), as in Phase 8
GRADIO_SHARE    = True        # also print Gradio's own https link (*.gradio.live) for the recorder
LAUNCH_RECORDER = True

### 📘 1.3 Install pinned dependencies

**Purpose:** Install Phase 7's `requirements-api.txt` (no `-U`), add `python-multipart` (needed for audio uploads) to it, install the UI requirements if present, and check ffmpeg (decodes browser recordings).

**Expected Results:** ✅ for the pins (chromadb 1.5.9, sentence-transformers 5.7.0, peft 0.21.2, torchao ≥ 0.16), gradio and python-multipart present, ffmpeg found.

**Parameters:** none

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError
if not REPO_LINK.exists():
    REPO_LINK.symlink_to(DRIVE_PROJECT, target_is_directory=True)
assert (REPO_LINK / RAG_CONFIG_REL).exists(), f"rag_config not found under {DRIVE_PROJECT}"

req = next((p for p in [REPO_LINK / "requirements-api.txt", REPO_LINK / "src/api/requirements-api.txt"] if p.exists()), None)
assert req, "requirements-api.txt (Phase 7) not found"
lines = req.read_text().splitlines()
if not any(l.strip().lower().startswith("python-multipart") for l in lines):
    req.write_text("\n".join(lines + ["python-multipart>=0.0.9   # Phase 10: audio uploads (FastAPI File/Form)"]) + "\n")
    print("added python-multipart to", req.name)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
ui_req = next((p for p in [REPO_LINK / "requirements-ui.txt", REPO_LINK / "src/ui/requirements-ui.txt"] if p.exists()), None)
if ui_req:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ui_req)], check=True)

PINS = {"chromadb": "==1.5.9", "sentence-transformers": "==5.7.0", "peft": "==0.21.2", "torchao": ">=0.16"}
def _ok(v, spec):
    if v is None: return False
    if spec.startswith("=="): return v == spec[2:]
    want = tuple(int(x) for x in spec[2:].split("."))
    return tuple(int(x) for x in v.split("+")[0].split(".")[:len(want)] if x.isdigit()) >= want
for pkg, spec in PINS.items():
    try: v = version(pkg)
    except PackageNotFoundError: v = None
    print(f"{'✅' if _ok(v, spec) else '❌'} {pkg:22s} {v}  (want {spec})")
for pkg in ("transformers", "torch", "fastapi", "python-multipart", "gradio", "uvicorn"):
    try: print(f"   {pkg:22s} {version(pkg)}")
    except PackageNotFoundError: print(f"❌ {pkg:22s} MISSING")
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False)
print("ffmpeg:", shutil.which("ffmpeg") or "missing (WAV-only fallback)")

### 📘 1.4 Helpers, CHECKS registry, parser and eval set

**Purpose:** Shared helpers, the `CHECKS` dict, an https tunnel helper for the browser microphone, the Phase 6 rule parser (used to compare the filters parsed from spoken vs typed queries) and the Phase 9 eval set.

**Expected Results:** The eval file and number of labelled queries.

**Parameters:** `EVAL_PATH`.

In [ ]:
import json, re, random, importlib, glob
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Audio
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 220)
if str(REPO_LINK) not in sys.path: sys.path.insert(0, str(REPO_LINK))
RESULTS_DIR.mkdir(parents=True, exist_ok=True); LOG_DIR.mkdir(parents=True, exist_ok=True)

CHECKS = {}
_ICON = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌", "PENDING": "⏳", "SKIP": "⏭️"}
def check(name, status, detail=""):
    CHECKS[name] = {"status": status, "detail": str(detail)}
    print(f"{_ICON.get(status, '•')} {name}: {status} — {detail}")
    (RESULTS_DIR / "checks_progress.json").write_text(json.dumps(CHECKS, indent=2))

def save_json(obj, name):
    (RESULTS_DIR / name).write_text(json.dumps(obj, indent=2, default=str))

def start_tunnel(port, log_name):
    # https URL for a local port via a Cloudflare quick tunnel (same approach as Phase 8). Returns the URL or None.
    if not USE_CLOUDFLARED:
        return None
    exe = "/content/cloudflared"
    if not os.path.exists(exe):
        subprocess.run(["wget", "-q", "-O", exe,
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod(exe, 0o755)
    log = LOG_DIR / f"{log_name}.log"
    proc = subprocess.Popen([exe, "tunnel", "--url", f"http://127.0.0.1:{port}", "--protocol", "http2", "--no-autoupdate"],
                            stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log.read_text(errors="ignore"))
        if m: return m.group(0)
        time.sleep(1)
    print("⚠️ tunnel URL not found; see", log); return None

RAG_CFG = json.loads((REPO_LINK / RAG_CONFIG_REL).read_text())
from src.rag.query_parser import RuleBasedParser, ProductTypeLexicon
_brands = json.load(open(REPO_LINK / RAG_CFG.get("brands_file", "src/rag/brands.json")))
PARSER = RuleBasedParser(known_brands=_brands, type_lexicon=ProductTypeLexicon.load(str(REPO_LINK / RAG_CFG["type_lexicon_file"])))

import src.eval.metrics as M
def find_eval_file():
    if EVAL_PATH: return REPO_LINK / EVAL_PATH
    p9 = PHASE9_DIR / "summary.json"          # Phase 9 recorded which file it used
    if p9.exists():
        f = Path(json.loads(p9.read_text()).get("eval_file", ""))
        if f.exists(): return f
    for pat in ["data/eval*/eval_queries_expanded.json", "data/*/eval_queries_expanded.json", "data/eval_queries_expanded.json",
                "eval*/eval_queries_expanded.json", "results/phase6/eval_queries_expanded.json",
                "src/rag/eval_queries_expanded.json", "data/eval*/*.json", "data/*/*eval*.json*"]:
        hits = sorted(REPO_LINK.glob(pat))
        if hits: return hits[0]
    raise FileNotFoundError("eval set not found — set EVAL_PATH in 1.2")
EVAL_FILE = find_eval_file()
EVALSET = M.normalize_eval_set(M.load_records(EVAL_FILE))
QBY = {e["qid"]: e for e in EVALSET}
print(f"eval set: {EVAL_FILE.relative_to(REPO_LINK)} — {len(EVALSET)} queries, {sum(bool(e['qrels']) for e in EVALSET)} labelled")

### 📘 1.5 Back up the files Phase 10 changes

**Purpose:** Copy the Phase 7–8 versions of the API and UI files to `src/_backup_before_phase10/` once, so you can roll back.

**Expected Results:** A list of backed-up files (skipped if the backup already exists).

**Parameters:** none

In [ ]:
BACKUP = REPO_LINK / "src" / "_backup_before_phase10"
CHANGED = ["api/main.py", "api/schemas.py", "api/service.py", "ui/app.py", "ui/client.py", "ui/render.py"]
if BACKUP.exists():
    print("backup already exists:", sorted(p.name for p in BACKUP.rglob("*.py")))
else:
    for rel in CHANGED:
        src = REPO_LINK / "src" / rel
        if src.exists():
            dst = BACKUP / rel; dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
    print("backed up:", [str(p.relative_to(BACKUP)) for p in BACKUP.rglob("*.py")])

## 2. Code (`%%writefile` into the repo)

### 📘 2.0 Package folders

**Purpose:** Create `src/asr/` and `src/tts/`.

**Expected Results:** `ok`.

**Parameters:** none

In [ ]:
for d in ("asr", "tts"): (REPO_LINK / "src" / d).mkdir(parents=True, exist_ok=True)
print("ok")

### 📘 2.1 `src/asr/__init__.py`

**Purpose:** package marker

**Expected Results:** `Writing /content/ShopTalk/src/asr/__init__.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/asr/__init__.py
"""Speech-to-text for ShopTalk (Phase 10)."""

### 📘 2.2 `src/asr/audio.py`

**Purpose:** decode any recording (wav/webm/ogg/mp3) to 16 kHz mono with ffmpeg; stdlib WAV fallback; 30 s limit

**Expected Results:** `Writing /content/ShopTalk/src/asr/audio.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/asr/audio.py
"""Audio decoding for speech-to-text (Phase 10).

Every input (browser recording, uploaded file, WAV from the notebook) is turned into 16 kHz mono float32,
which is what Whisper expects. ffmpeg handles any container/codec (webm, ogg, mp3, m4a, wav); a pure-stdlib
WAV reader is the fallback when ffmpeg is not installed (e.g. a slim Docker image in Phase 11).
"""
from __future__ import annotations

import io
import shutil
import subprocess
import wave
from pathlib import Path
from typing import Union

import numpy as np

SAMPLE_RATE = 16000
AudioInput = Union[bytes, str, Path]


class AudioError(ValueError):
    """The audio could not be decoded, or is empty / too long."""


def _read_bytes(src: AudioInput) -> bytes:
    if isinstance(src, (str, Path)):
        return Path(src).read_bytes()
    return bytes(src)


def _ffmpeg_decode(data: bytes, sr: int) -> np.ndarray:
    cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-i", "pipe:0",
           "-ac", "1", "-ar", str(sr), "-f", "f32le", "pipe:1"]
    proc = subprocess.run(cmd, input=data, capture_output=True, timeout=60)
    if proc.returncode != 0:
        raise AudioError(f"ffmpeg could not decode the audio: {proc.stderr.decode(errors='ignore')[-300:]}")
    return np.frombuffer(proc.stdout, dtype=np.float32).copy()


def _wav_decode(data: bytes, sr: int) -> np.ndarray:
    try:
        with wave.open(io.BytesIO(data)) as w:
            n_ch, width, rate, n = w.getnchannels(), w.getsampwidth(), w.getframerate(), w.getnframes()
            raw = w.readframes(n)
    except (wave.Error, EOFError) as e:
        raise AudioError(f"not a PCM WAV file and ffmpeg is not installed: {e}") from e
    if width == 2:
        x = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
    elif width == 4:
        x = np.frombuffer(raw, dtype="<i4").astype(np.float32) / 2147483648.0
    elif width == 1:
        x = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    else:
        raise AudioError(f"unsupported WAV sample width: {width} bytes")
    if n_ch > 1:
        x = x.reshape(-1, n_ch).mean(axis=1)
    if rate != sr and len(x):
        t_new = np.arange(int(round(len(x) * sr / rate))) / sr
        x = np.interp(t_new, np.arange(len(x)) / rate, x).astype(np.float32)
    return x


def decode_audio(src: AudioInput, sr: int = SAMPLE_RATE, max_seconds: float = 30.0) -> tuple:
    """Returns (float32 mono waveform at `sr`, duration in seconds). Raises AudioError."""
    data = _read_bytes(src)
    if not data:
        raise AudioError("empty audio")
    x = _ffmpeg_decode(data, sr) if shutil.which("ffmpeg") else _wav_decode(data, sr)
    seconds = len(x) / sr
    if seconds < 0.1:
        raise AudioError("audio is shorter than 0.1 s")
    if max_seconds and seconds > max_seconds:
        raise AudioError(f"audio is {seconds:.1f} s; the limit is {max_seconds:.0f} s")
    peak = float(np.abs(x).max()) if len(x) else 0.0
    if peak > 1.0:
        x = x / peak
    return x.astype(np.float32), seconds


def to_wav_bytes(x: np.ndarray, sr: int = SAMPLE_RATE) -> bytes:
    """float waveform -> 16-bit PCM WAV bytes (stdlib only)."""
    x = np.clip(np.asarray(x, dtype=np.float32), -1.0, 1.0)
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(sr)
        w.writeframes((x * 32767.0).astype("<i2").tobytes())
    return buf.getvalue()


def write_wav(path: Union[str, Path], x: np.ndarray, sr: int = SAMPLE_RATE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(to_wav_bytes(x, sr))
    return path

### 📘 2.3 `src/asr/whisper_asr.py`

**Purpose:** `WhisperASR` (model + processor, fp16 on GPU, loaded once), `FakeASR`, `build_asr()` from env / rag_config

**Expected Results:** `Writing /content/ShopTalk/src/asr/whisper_asr.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/asr/whisper_asr.py
"""Speech-to-text for ShopTalk (Phase 10): open-source Whisper models via Hugging Face transformers.

Loaded once at API startup and kept on the GPU next to the LLM. Uses the model + processor directly
(not `pipeline()`), which works the same for whisper-tiny/base/small(.en), distil-whisper and
whisper-large-v3-turbo.

    asr = WhisperASR("openai/whisper-base.en")
    out = asr.transcribe(waveform_16k)   # {"text", "ms", "audio_s", "model"}
"""
from __future__ import annotations

import os
import time
from typing import Optional

import numpy as np

from .audio import SAMPLE_RATE

DEFAULT_MODEL = "openai/whisper-base.en"


class WhisperASR:
    def __init__(self, model_id: str = DEFAULT_MODEL, device: Optional[str] = None, fp16: bool = True,
                 language: str = "en", max_new_tokens: int = 96):
        import torch
        from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

        self.model_id = model_id
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.float16 if (fp16 and self.device.startswith("cuda")) else torch.float32
        self.language = language
        self.max_new_tokens = max_new_tokens
        # English-only checkpoints (*.en) reject the language/task arguments
        self.multilingual = not model_id.rstrip("/").endswith(".en")
        t0 = time.perf_counter()
        self.processor = AutoProcessor.from_pretrained(model_id)
        try:
            model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, dtype=self.dtype, low_cpu_mem_usage=True)
        except TypeError:   # older transformers: torch_dtype
            model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, torch_dtype=self.dtype, low_cpu_mem_usage=True)
        self.model = model.to(self.device).eval()
        self.load_s = time.perf_counter() - t0
        self._torch = torch

    def _sync(self):
        if self.device.startswith("cuda"):
            self._torch.cuda.synchronize()

    def transcribe(self, audio: np.ndarray, sr: int = SAMPLE_RATE) -> dict:
        if sr != SAMPLE_RATE:
            raise ValueError(f"expected {SAMPLE_RATE} Hz audio, got {sr}")
        torch = self._torch
        self._sync()
        t0 = time.perf_counter()
        feats = self.processor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt")
        inputs = feats.input_features.to(self.device, self.dtype)
        kwargs = {"max_new_tokens": self.max_new_tokens}
        if self.multilingual:
            kwargs.update(language=self.language, task="transcribe")
        with torch.inference_mode():
            ids = self.model.generate(inputs, **kwargs)
        text = self.processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        self._sync()
        return {"text": text, "ms": round((time.perf_counter() - t0) * 1000, 1),
                "audio_s": round(len(audio) / SAMPLE_RATE, 2), "model": self.model_id}

    def gpu_mb(self) -> Optional[float]:
        if not self.device.startswith("cuda"):
            return None
        return round(sum(p.numel() * p.element_size() for p in self.model.parameters()) / 1e6, 1)

    def unload(self):
        del self.model
        if self.device.startswith("cuda"):
            self._torch.cuda.empty_cache()


class FakeASR:
    """No-GPU stand-in (SHOPTALK_FAKE=1): returns SHOPTALK_FAKE_TRANSCRIPT, or a fixed sentence."""
    model_id = "fake-asr"
    load_s = 0.0

    def transcribe(self, audio: np.ndarray, sr: int = SAMPLE_RATE) -> dict:
        t0 = time.perf_counter()
        text = os.environ.get("SHOPTALK_FAKE_TRANSCRIPT", "Show me a red shirt for men under $50")
        return {"text": text, "ms": round((time.perf_counter() - t0) * 1000, 1),
                "audio_s": round(len(audio) / sr, 2), "model": self.model_id}


def build_asr(config: Optional[dict] = None):
    """ASR from env / rag_config['asr'].  Env wins: SHOPTALK_ASR_MODEL, SHOPTALK_ASR_DEVICE."""
    if os.environ.get("SHOPTALK_FAKE") == "1":
        return FakeASR()
    cfg = (config or {}).get("asr", {}) or {}
    return WhisperASR(os.environ.get("SHOPTALK_ASR_MODEL") or cfg.get("model_id") or DEFAULT_MODEL,
                      device=os.environ.get("SHOPTALK_ASR_DEVICE") or cfg.get("device"),
                      max_new_tokens=int(cfg.get("max_new_tokens", 96)))

### 📘 2.4 `src/asr/text_norm.py`

**Purpose:** transcript → query (spoken numbers → digits, "30 dollars" → "$30"), WER, key-word recall, parsed-filter agreement, `speakable()` for TTS

**Expected Results:** `Writing /content/ShopTalk/src/asr/text_norm.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/asr/text_norm.py
"""Text normalisation and speech-to-text metrics (Phase 10).

* `normalize` makes a reference and a transcript comparable: lower case, no punctuation, "$30" ->
  "30 dollars", number words -> digits ("thirty" -> 30, "nine point five" -> 9.5), "men's" -> "mens".
* `wer` / `corpus_wer`: word error rate on normalised text.
* `key_terms` / `key_term_recall`: the words that change search results (brands, numbers, colours,
  product type) and whether the transcript kept them.
* `filter_agreement`: run the ShopTalk query parser on the reference and on the transcript and compare
  the filters it extracts (price, brand, colour, size, gender, product type). This is the most direct
  "did mishearing change the search?" measure.
* `speakable`: the reverse direction for text-to-speech ("$11.99" -> "eleven dollars ninety nine").
"""
from __future__ import annotations

import re
from typing import Iterable, Optional

_ONES = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven",
         "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
_TENS = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]


# ----------------------------------------------------------------------------- numbers -> words (TTS)
def int_to_words(n: int) -> str:
    if n < 0:
        return "minus " + int_to_words(-n)
    if n < 20:
        return _ONES[n]
    if n < 100:
        return _TENS[n // 10] + ("" if n % 10 == 0 else " " + _ONES[n % 10])
    if n < 1000:
        return _ONES[n // 100] + " hundred" + ("" if n % 100 == 0 else " " + int_to_words(n % 100))
    if n < 1_000_000:
        return int_to_words(n // 1000) + " thousand" + ("" if n % 1000 == 0 else " " + int_to_words(n % 1000))
    return " ".join(_ONES[int(d)] for d in str(n))


def _decimal_words(s: str) -> str:
    whole, _, frac = s.partition(".")
    out = int_to_words(int(whole or 0))
    return out + (" point " + " ".join(_ONES[int(d)] for d in frac) if frac else "")


def speakable(text: str) -> str:
    """Text for a TTS voice that cannot read digits: removes [ID: …] tags and markdown, spells numbers."""
    t = re.sub(r"\[\s*ID\s*:[^\]]*\]", "", text or "")
    t = re.sub(r"[*_`#>|]", " ", t)

    def money(m):
        whole = int(m.group(1).replace(",", ""))
        cents = int((m.group(2) or "0").ljust(2, "0")[:2])
        s = int_to_words(whole) + (" dollar" if whole == 1 else " dollars")
        return s + (f" {int_to_words(cents)}" if cents else "")

    t = re.sub(r"\$\s?(\d[\d,]*)(?:\.(\d{1,2}))?", money, t)
    t = re.sub(r"(\d+)\s*%", lambda m: int_to_words(int(m.group(1))) + " percent", t)
    t = re.sub(r"\d+(?:\.\d+)?", lambda m: _decimal_words(m.group()), t)
    return re.sub(r"\s+", " ", t).strip()


# ----------------------------------------------------------------------------- words -> numbers (WER)
_UNITS = {w: i for i, w in enumerate(_ONES)}            # zero..nineteen
_TENS_W = {w: i * 10 for i, w in enumerate(_TENS) if w}  # twenty..ninety


def _is_num_word(t: str) -> bool:
    return t in _UNITS or t in _TENS_W


def _parse_run(tokens: list, i: int) -> tuple:
    """Parse one spoken number starting at tokens[i]. Returns (value_str, next_index)."""
    total, cur, last, j = 0, 0, None, i
    while j < len(tokens):
        t = tokens[j]
        if t == "a" and j + 1 < len(tokens) and tokens[j + 1] in ("hundred", "thousand") and last is None:
            cur, last = 1, "unit"
        elif t in _UNITS:
            v = _UNITS[t]
            if last in ("unit", "teen") or (last == "tens" and v >= 10):
                break
            cur += v
            last = "teen" if v >= 10 else "unit"
        elif t in _TENS_W:
            if last in ("unit", "teen", "tens"):
                break
            cur += _TENS_W[t]
            last = "tens"
        elif t == "hundred" and last in ("unit", "teen"):
            cur *= 100
            last = "hundred"
        elif t == "thousand" and (cur or total):
            total, cur, last = total + max(cur, 1) * 1000, 0, "thousand"
        elif t == "and" and last in ("hundred", "thousand") and j + 1 < len(tokens) and _is_num_word(tokens[j + 1]):
            pass
        else:
            break
        j += 1
    value = str(total + cur)
    # decimals: "nine point five" -> 9.5
    if j + 1 < len(tokens) and tokens[j] == "point" and tokens[j + 1] in _UNITS and _UNITS[tokens[j + 1]] < 10:
        digits, k = [], j + 1
        while k < len(tokens) and tokens[k] in _UNITS and _UNITS[tokens[k]] < 10:
            digits.append(str(_UNITS[tokens[k]]))
            k += 1
        value, j = value + "." + "".join(digits), k
    return value, j


def _words_to_numbers(tokens: list) -> list:
    out, i = [], 0
    while i < len(tokens):
        t = tokens[i]
        starts = _is_num_word(t) or (t == "a" and i + 1 < len(tokens) and tokens[i + 1] in ("hundred", "thousand"))
        if starts:
            value, j = _parse_run(tokens, i)
            if j > i:
                out.append(value)
                i = j
                continue
        out.append(t)
        i += 1
    return out


_TENS_RE = "twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety"
_UNITS_RE = "one|two|three|four|five|six|seven|eight|nine"


def transcript_to_query(text: str) -> str:
    """Speech-to-text output -> the query the pipeline sees: spelled numbers become digits and
    "30 dollars" becomes "$30", so prices and sizes reach the parser the same way as typed queries
    (the parser reads "$30"/"under 30" but not "thirty", and the product-type lexicon even maps
    "thirty" to HAT). Everything else is kept as transcribed."""
    t = re.sub(rf"\b({_TENS_RE})-({_UNITS_RE})\b", r"\1 \2", text or "", flags=re.I)
    chunks = t.split()
    cores = [re.sub(r"^[^\w$]+|[^\w]+$", "", c).lower() for c in chunks]
    out, i = [], 0
    while i < len(chunks):
        c = cores[i]
        starts = _is_num_word(c) or (c == "a" and i + 1 < len(cores) and cores[i + 1] in ("hundred", "thousand"))
        if starts:
            # a run ends at a chunk carrying punctuation ("thirty," "dollars.")
            stop = i
            while stop < len(chunks) - 1 and chunks[stop].lower() == cores[stop]:
                stop += 1
            value, j = _parse_run(cores[: stop + 1], i)
            if j > i and not (value == "1" and c == "one" and j == i + 1):   # keep "one" alone ("one piece")
                tail = re.sub(r"^.*?([^\w]*)$", r"\1", chunks[j - 1])
                out.append(value + tail)
                i = j
                continue
        out.append(chunks[i])
        i += 1
    q = " ".join(out)
    q = re.sub(r"\b(\d+(?:\.\d+)?)\s*(?:us\s+)?(?:dollars?|bucks?)\b", r"$\1", q, flags=re.I)
    return q


def normalize(text: str) -> str:
    t = (text or "").lower().replace("’", "'").replace("‘", "'")
    t = re.sub(r"\$\s?(\d[\d,]*(?:\.\d+)?)", lambda m: m.group(1).replace(",", "") + " dollars", t)
    t = re.sub(r"(\d),(\d{3})", r"\1\2", t)
    t = t.replace("'s ", "s ").replace("'", "")
    t = re.sub(r"(?<!\d)\.|\.(?!\d)", " ", t)          # keep decimal points only
    t = re.sub(r"[^a-z0-9.\s]", " ", t)
    t = re.sub(r"(\d)([a-z])", r"\1 \2", t)            # "10in" -> "10 in"
    t = re.sub(r"([a-z])(\d)", r"\1 \2", t)
    tokens = _words_to_numbers(t.split())
    tokens = [{"dollar": "dollars", "bucks": "dollars", "buck": "dollars"}.get(x, x) for x in tokens]
    return " ".join(tokens)


# ----------------------------------------------------------------------------- WER
def edit_counts(ref: list, hyp: list) -> dict:
    """Levenshtein alignment counts: substitutions, deletions, insertions."""
    n, m = len(ref), len(hyp)
    d = [[(0, 0, 0, 0)] * (m + 1) for _ in range(n + 1)]   # (cost, S, D, I)
    for i in range(1, n + 1):
        d[i][0] = (i, 0, i, 0)
    for j in range(1, m + 1):
        d[0][j] = (j, 0, 0, j)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i - 1] == hyp[j - 1]:
                d[i][j] = d[i - 1][j - 1]
                continue
            s, de, ins = d[i - 1][j - 1], d[i - 1][j], d[i][j - 1]
            d[i][j] = min((s[0] + 1, s[1] + 1, s[2], s[3]), (de[0] + 1, de[1], de[2] + 1, de[3]),
                          (ins[0] + 1, ins[1], ins[2], ins[3] + 1))
    c = d[n][m]
    return {"S": c[1], "D": c[2], "I": c[3], "N": n}


def wer(reference: str, hypothesis: str) -> float:
    r, h = normalize(reference).split(), normalize(hypothesis).split()
    c = edit_counts(r, h)
    return (c["S"] + c["D"] + c["I"]) / max(c["N"], 1)


def corpus_wer(pairs: Iterable[tuple]) -> float:
    errs = n = 0
    for ref, hyp in pairs:
        c = edit_counts(normalize(ref).split(), normalize(hyp).split())
        errs += c["S"] + c["D"] + c["I"]
        n += c["N"]
    return errs / max(n, 1)


# ----------------------------------------------------------------------------- what matters for search
def key_terms(reference: str, parser=None) -> list:
    """Normalised words that change the search: numbers + brand / colour / product-type words."""
    norm = normalize(reference)
    terms = [t for t in norm.split() if re.fullmatch(r"\d+(\.\d+)?", t)]
    if parser is not None:
        pq = parser.parse(reference)
        for v in [pq.brand, getattr(pq, "type_term", None), *(pq.colors or [])]:
            if v:
                terms += normalize(str(v)).split()
    return list(dict.fromkeys(_stem(t) for t in terms))


def key_term_recall(reference: str, hypothesis: str, parser=None) -> Optional[float]:
    terms = key_terms(reference, parser)
    if not terms:
        return None
    hyp = {_stem(t) for t in normalize(hypothesis).split()}
    return sum(_stem(t) in hyp for t in terms) / len(terms)


def _stem(t: str) -> str:
    """Plural-insensitive match: the parser reports 'sandal' for 'sandals'."""
    return t[:-1] if len(t) > 3 and t.endswith("s") and not t.endswith("ss") else t


FILTER_FIELDS = ("brand", "min_price", "max_price", "colors", "gender", "size", "product_types")


def _canon(v):
    if isinstance(v, (list, tuple, set)):
        return tuple(sorted(str(x).lower() for x in v))
    if isinstance(v, str):
        return normalize(v)
    return v


def filter_agreement(reference: str, hypothesis: str, parser) -> dict:
    """Per-field equality of the parsed filters; 'all' is True when every field matches."""
    a, b = parser.parse(reference), parser.parse(hypothesis)
    out = {f: _canon(getattr(a, f, None)) == _canon(getattr(b, f, None)) for f in FILTER_FIELDS}
    out["all"] = all(out.values())
    out["mismatched"] = [f for f in FILTER_FIELDS if not out[f]]
    return out

### 📘 2.5 `src/tts/__init__.py`

**Purpose:** package marker

**Expected Results:** `Writing /content/ShopTalk/src/tts/__init__.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/tts/__init__.py
"""Optional text-to-speech for the ShopTalk UI (Phase 10)."""

### 📘 2.6 `src/tts/mms_tts.py`

**Purpose:** optional read-aloud voice: Meta MMS (VITS) via transformers on CPU, numbers spelled out, IDs removed

**Expected Results:** `Writing /content/ShopTalk/src/tts/mms_tts.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/tts/mms_tts.py
"""Optional text-to-speech for the ShopTalk UI (Phase 10): reads the answer aloud.

Default voice: facebook/mms-tts-eng (Meta MMS, a small VITS model, ~36M parameters) through Hugging Face
transformers, so it adds no new packages to the pinned API environment. It runs on CPU by default, so it
never waits for the GPU lock held by the LLM. Licence: check the model card before any commercial use
(MMS checkpoints are published for non-commercial use); `SHOPTALK_TTS_MODEL` swaps in another VITS
checkpoint whose tokenizer does not need a phonemizer.

The MMS English tokenizer only knows letters, so text goes through `speakable()` first: product IDs are
removed and numbers/prices are spelled out ("$11.99" -> "eleven dollars ninety nine").
"""
from __future__ import annotations

import os
import time
from typing import Optional

import numpy as np

from src.asr.audio import to_wav_bytes
from src.asr.text_norm import speakable

DEFAULT_TTS_MODEL = "facebook/mms-tts-eng"
MAX_CHARS = 600


class MMSTTS:
    def __init__(self, model_id: str = DEFAULT_TTS_MODEL, device: Optional[str] = None, seed: int = 0):
        import torch
        from transformers import AutoTokenizer, VitsModel

        self.model_id = model_id
        self.device = device or "cpu"
        self.seed = seed
        t0 = time.perf_counter()
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = VitsModel.from_pretrained(model_id).to(self.device).eval()
        self.sample_rate = int(self.model.config.sampling_rate)
        self.load_s = time.perf_counter() - t0
        self._torch = torch

    def synthesize(self, text: str) -> dict:
        """-> {"wav": bytes, "ms": float, "audio_s": float, "text": spoken text}"""
        torch = self._torch
        spoken = speakable(text)[:MAX_CHARS]
        if not spoken:
            raise ValueError("nothing to say")
        t0 = time.perf_counter()
        torch.manual_seed(self.seed)          # VITS samples durations: fixed seed = same voice every time
        inputs = self.tokenizer(spoken, return_tensors="pt").to(self.device)
        with torch.inference_mode():
            wave = self.model(**inputs).waveform[0].float().cpu().numpy()
        ms = (time.perf_counter() - t0) * 1000
        return {"wav": to_wav_bytes(wave, self.sample_rate), "ms": round(ms, 1),
                "audio_s": round(len(wave) / self.sample_rate, 2), "text": spoken, "sample_rate": self.sample_rate}


class FakeTTS:
    """No-model stand-in (SHOPTALK_FAKE=1): a short 440 Hz beep per sentence."""
    model_id = "fake-tts"
    sample_rate = 16000
    load_s = 0.0

    def synthesize(self, text: str) -> dict:
        spoken = speakable(text)
        if not spoken:
            raise ValueError("nothing to say")
        t = np.arange(int(0.3 * self.sample_rate)) / self.sample_rate
        wave = 0.2 * np.sin(2 * np.pi * 440 * t)
        return {"wav": to_wav_bytes(wave, self.sample_rate), "ms": 0.1, "audio_s": 0.3, "text": spoken,
                "sample_rate": self.sample_rate}


def tts_enabled() -> bool:
    return os.environ.get("SHOPTALK_TTS", "0") == "1"


def build_tts():
    if os.environ.get("SHOPTALK_FAKE") == "1":
        return FakeTTS()
    return MMSTTS(os.environ.get("SHOPTALK_TTS_MODEL", DEFAULT_TTS_MODEL),
                  device=os.environ.get("SHOPTALK_TTS_DEVICE", "cpu"))

### 📘 2.7 `src/api/schemas.py`

**Purpose:** adds `TranscribeResponse`, `AudioRecommendResponse`, `SpeakRequest`, and asr/tts fields on `/health`

**Expected Results:** `Writing /content/ShopTalk/src/api/schemas.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/api/schemas.py
"""Request / response models for the ShopTalk inference API (Phase 7, voice added in Phase 10).

The API takes plain text (`query`), so speech-to-text can be added in front later (Phase 10)
without changing the pipeline. `history` is optional: send back the `history_entry` of earlier
responses to ask follow-ups such as "something similar but cheaper".
"""
from __future__ import annotations

from typing import List, Literal, Optional

from pydantic import BaseModel, Field


class HistoryTurn(BaseModel):
    """
    Represents a single conversational turn in the history stack.

    Attributes:
        parsed (dict): The resolved query filters and metadata from the query parser.
        products (List[dict]): List of product dictionary representation cards shown in this turn.
    """
    parsed: dict
    products: List[dict] = Field(default_factory=list)


class RecommendRequest(BaseModel):
    """
    Inbound payload validation schema for the recommendation endpoint.

    Attributes:
        query (str): Plain-text natural language user inquiry. Constrained between 1 and 500 characters.
        history (List[HistoryTurn]): Accumulated historical context turns to assist in follow-up queries.
        session_id (str, optional): Tracking token utilized for logging telemetry cohorts.
    """
    query: str = Field(..., min_length=1, max_length=500, examples=["Show me red sneakers for men under $50"])
    history: List[HistoryTurn] = Field(default_factory=list, description="Earlier turns (last 3 are used)")
    session_id: Optional[str] = Field(None, max_length=100, description="Optional client session id, logged")


class ProductCard(BaseModel):
    """
    Standardized representation schema of an individual product returned to the UI client.

    Attributes:
        id (str): Unique product item identifier.
        title (str): Title/name of the product.
        brand (str, optional): Product brand metadata.
        price (float, optional): Sale price in USD.
        color (str, optional): Identified primary color attribute.
        product_type (str, optional): Normalized taxonomy category name.
        image_url (str, optional): Valid URL referencing the product photo thumbnail.
        link (str, optional): Target product URL redirecting via /click endpoint.
        score (float): Vector search match confidence similarity score.
        matched (dict): Boolean validation flags reflecting specific search criteria checks.
        price_unknown (bool): Flag showing if the product does not have pricing details.
    """
    id: str
    title: str = ""
    brand: Optional[str] = None
    price: Optional[float] = None
    color: Optional[str] = None
    product_type: Optional[str] = None
    image_url: Optional[str] = None
    link: Optional[str] = None
    score: float = 0.0
    matched: dict = Field(default_factory=dict)
    price_unknown: bool = False


class RecommendResponse(BaseModel):
    """
    Outbound payload structure returned by the recommendation engine.

    Attributes:
        request_id (str): Unique hash tracking token for this request instance.
        query (str): Mirror of the validated user query.
        answer (str): Fully cited, grounded natural language recommendation response.
        products (List[ProductCard]): Evaluated and filtered catalog recommendation cards.
        no_match (bool): Indicator flag if no matching products were returned by retriever.
        unmet_constraints (List[str]): List of identified filters that could not be satisfied.
        parsed (dict): The parsed query parameters generated by the semantic extraction model.
        grounding (dict): Evaluation tracking keys for hallucinated or cited records.
        llm (str, optional): Identifier of the active inference LLM model.
        latency_ms (dict): Millisecond timing intervals spent inside each processing phase.
        history_entry (HistoryTurn): Packaged history block reflecting current turn outputs.
    """
    request_id: str
    query: str
    answer: str
    products: List[ProductCard]
    no_match: bool
    unmet_constraints: List[str] = Field(default_factory=list,
                                         description="Requested attributes none of the products has, e.g. ['type']")
    parsed: dict
    grounding: dict
    llm: Optional[str] = None
    latency_ms: dict
    history_entry: HistoryTurn


class FeedbackRequest(BaseModel):
    """
    Explicit feedback logging schema capturing user-level satisfaction signals.

    Attributes:
        request_id (str): The unique request hash corresponding to the logged turn.
        rating (str): Directional evaluation status, strictly 'up' or 'down'.
        comment (str, optional): Free-form review statement.
        session_id (str, optional): Associated tracking session context.
    """
    request_id: str = Field(..., max_length=64)
    rating: Literal["up", "down"]
    comment: Optional[str] = Field(None, max_length=1000)
    session_id: Optional[str] = Field(None, max_length=100)


class HealthResponse(BaseModel):
    """
    Server status payload reporting active component states.

    Attributes:
        status (str): Current operating state ('ok', 'loading', 'error').
        llm (str, optional): Active language model signature string.
        embed_model (str, optional): Active embedding model signature string.
        index_size (int, optional): Size of the local vector database collection.
        uptime_s (float): Total runtime in seconds since the service was launched.
        error (str, optional): Exception context if status evaluates to 'error'.
    """
    status: Literal["ok", "loading", "error"]
    llm: Optional[str] = None
    embed_model: Optional[str] = None
    index_size: Optional[int] = None
    uptime_s: float
    error: Optional[str] = None
    asr: Optional[str] = Field(None, description="Speech-to-text model, when loaded (Phase 10)")
    asr_status: Optional[str] = Field(None, description="'ok' | 'loading' | 'error' | 'disabled'")
    tts: Optional[str] = Field(None, description="Text-to-speech model, when enabled (Phase 10)")
    tts_status: Optional[str] = None


# ---------------- Phase 10: voice ----------------
class TranscribeResponse(BaseModel):
    """
    Result of POST /transcribe (speech -> query text).

    Attributes:
        request_id (str): Unique id of this transcription (logged in asr.jsonl).
        text (str): The query to search: the transcript with spelled numbers turned into digits
            ("under thirty dollars" -> "under $30"). Shown to the user before searching.
        raw_text (str): The model's transcript before that normalisation.
        model (str): Speech-to-text model id.
        audio_s (float): Length of the audio in seconds.
        latency_ms (dict): decode (audio -> 16 kHz), queue (waiting for the GPU) and asr (model) times.
    """
    request_id: str
    text: str
    raw_text: str
    model: str
    audio_s: float
    latency_ms: dict


class AudioRecommendResponse(RecommendResponse):
    """POST /recommend/audio: the normal /recommend response plus the transcript that was searched."""
    transcript: str = Field(..., description="The query that was searched (normalised transcript)")
    raw_transcript: str = Field(..., description="Speech-to-text output before number normalisation")
    asr_model: str
    audio_s: float


class SpeakRequest(BaseModel):
    """POST /speak: text to read aloud (product IDs and markdown are removed, numbers spelled out)."""
    text: str = Field(..., min_length=1, max_length=2000)

### 📘 2.8 `src/api/service.py`

**Purpose:** metrics also track `asr` and `tts` latency and voice request counts

**Expected Results:** `Writing /content/ShopTalk/src/api/service.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/api/service.py
"""Runtime pieces of the API that are independent of FastAPI: request logging, rolling latency
metrics, and the streaming version of ShopTalkPipeline.run().

Logs are JSON Lines, one record per line, so they can be read with pandas.read_json(lines=True).
They are the raw material for the feedback loop and for the Phase 9 evaluation:
  requests.jsonl  - every /recommend call: query, parsed filters, product IDs, answer, latency
  feedback.jsonl  - explicit feedback (thumbs up/down) and implicit feedback (product clicks)
"""
from __future__ import annotations

import json
import os
import threading
import time
from collections import deque
from datetime import datetime, timezone
from typing import Iterator, Optional

import numpy as np

from src.rag.generator import NO_MATCH_ANSWER, check_grounding, enforce_ids, remove_unshown_ids
from src.rag.pipeline import ShopTalkPipeline, unmet_constraints


def utc_now() -> str:
    """
    Generates an ISO-8601 formatted UTC timestamp.

    Returns:
        str: Timestamp string with millisecond precision.
    """
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds")


class JsonlLogger:
    """
    Append-only, thread-safe JSON Lines telemetry writer.

    Provides safe logging concurrency across background streaming worker threads.
    """

    def __init__(self, log_dir: str):
        """
        Initializes the logger workspace.

        Args:
            log_dir (str): Base destination directory for telemetry records.
        """
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)
        self._lock = threading.Lock()

    def write(self, name: str, record: dict) -> None:
        """
        Appends a dictionary entry into a JSONLines tracking log file.

        Args:
            name (str): Log prefix (e.g. 'requests' or 'feedback').
            record (dict): Context data dictionary payload to archive.
        """
        line = json.dumps({"ts": utc_now(), **record}, default=str, ensure_ascii=False)
        with self._lock, open(os.path.join(self.log_dir, f" {name}.jsonl".strip()), "a", encoding="utf-8") as f:
            f.write(line + "\n")


class Metrics:
    """
    Maintains in-memory counts and a rolling queue of query latencies.

    Handles metric calculations without writing database transactions.
    """

    STAGES = ("asr", "parse", "retrieve", "generate", "total", "tts")

    def __init__(self, window: int = 1000):
        """
        Initializes rolling metrics tracking.

        Args:
            window (int): The max size window of recent requests used for percentile calculations.
        """
        self._lock = threading.Lock()
        self.latencies = deque(maxlen=window)
        self.counts = {"requests": 0, "stream_requests": 0, "errors": 0, "no_match": 0,
                       "feedback_up": 0, "feedback_down": 0, "clicks": 0,
                       "transcribe": 0, "audio_requests": 0, "tts": 0}
        self.started = time.time()

    def inc(self, key: str, n: int = 1) -> None:
        """
        Safely increments an event counter.

        Args:
            key (str): Name of the counter.
            n (int): Value increment size.
        """
        with self._lock:
            self.counts[key] = self.counts.get(key, 0) + n

    def observe(self, latency_ms: dict) -> None:
        """
        Adds a new latency measurement to the rolling queue.

        Args:
            latency_ms (dict): Phase duration map in milliseconds.
        """
        with self._lock:
            self.latencies.append(latency_ms)

    def snapshot(self) -> dict:
        """
        Generates a metrics snapshot including P50, P95, and P99 percentiles.

        Returns:
            dict: Detailed running execution snapshot.
        """
        with self._lock:
            lat = list(self.latencies)
            counts = dict(self.counts)
        out = {"uptime_s": round(time.time() - self.started, 1), "counts": counts, "window": len(lat), "latency_ms": {}}
        for stage in self.STAGES:
            vals = [x[stage] for x in lat if stage in x]
            if vals:
                out["latency_ms"][stage] = {p: round(float(np.percentile(vals, q)), 1)
                                            for p, q in (("p50", 50), ("p95", 95), ("p99", 99))}
        return out


def log_record(request_id: str, session_id: Optional[str], endpoint: str, result: dict) -> dict:
    """
    Filters and formats query output into a standard logging payload structure.

    Args:
        request_id (str): Current query hash id.
        session_id (str): Current tracking session label.
        endpoint (str): Target web interface endpoint (e.g., '/recommend').
        result (dict): Complete result payload produced by pipeline.

    Returns:
        dict: Compact formatted record for disk storage.
    """
    p = result.get("parsed", {})
    return {
        "request_id": request_id, "session_id": session_id, "endpoint": endpoint,
        "query": result.get("query"),
        "parsed": {k: p.get(k) for k in ("semantic_query", "min_price", "max_price", "colors", "gender",
                                          "brand", "size", "product_types", "type_term", "type_unavailable",
                                          "is_followup") if p.get(k) not in (None, [], "", False)},
        "product_ids": [x["id"] for x in result.get("products", [])],
        "no_match": result.get("no_match"),
        "unmet_constraints": result.get("unmet_constraints", []),
        "answer": (result.get("answer") or "")[:1500],
        "hallucinated_ids_raw": result.get("grounding", {}).get("hallucinated_ids", []),
        "llm": result.get("llm"),
        "latency_ms": result.get("latency_ms"),
    }


def sse(event: str, data: dict) -> str:
    """
    Formats a data payload block into a standard Server-Sent Event text string.

    Args:
        event (str): Target action tag (e.g. 'token', 'meta').
        data (dict): Data payload to transmit.

    Returns:
        str: Standard formatted SSE transfer string.
    """
    return f"event: {event}\ndata: {json.dumps(data, default=str, ensure_ascii=False)}\n\n"


def run_stream(pipeline: ShopTalkPipeline, query: str, history: Optional[list] = None) -> Iterator[tuple]:
    """
    Streaming pipeline worker execution function.

    Yields intermediate and final pipeline outcomes as SSE compatible tuples.

    Args:
        pipeline (ShopTalkPipeline): Active model and DB execution structure.
        query (str): Plain-text request string.
        history (list, optional): Historical conversational turn states.

    Yields:
        tuple: Step label with correlated details dictionary.
    """
    t_start = time.perf_counter()
    pq = pipeline.parser.parse(query)
    pq, exclude = pipeline.resolve_followup(pq, history)
    is_followup_turn = bool(pq.is_followup and history)
    t_parse = time.perf_counter()

    retrieved = pipeline.retriever.retrieve(pq, top_k=pipeline.retrieve_k, exclude_ids=exclude,
                                            followup=is_followup_turn)
    context = retrieved[: pipeline.context_k]
    unmet = unmet_constraints(context)
    cards = [pipeline._product_card(p) for p in context]
    t_retr = time.perf_counter()
    yield "meta", {"parsed": pq.to_dict(), "products": cards, "no_match": not context,
                   "unmet_constraints": unmet,
                   "latency_ms": {"parse": round((t_parse - t_start) * 1000, 1),
                                  "retrieve": round((t_retr - t_parse) * 1000, 1)}}

    gen = pipeline.generator
    ttft = None
    if not context:
        answer = NO_MATCH_ANSWER
        yield "token", {"text": answer}
    else:
        gen_query = query if not is_followup_turn else f"{query} (previous request: {pq.semantic_query})"
        if hasattr(gen, "stream"):
            chunks = []
            for chunk in gen.stream(gen_query, context, parsed=pq, unmet=unmet):
                if ttft is None:
                    ttft = time.perf_counter() - t_retr
                chunks.append(chunk)
                yield "token", {"text": chunk}
            answer = "".join(chunks).strip()
        else:
            answer, _ = gen.generate(gen_query, context, parsed=pq, unmet=unmet)
            yield "token", {"text": answer}

    grounding = check_grounding(answer, context) if context else \
        {"cited_ids": [], "hallucinated_ids": [], "lists_ids": False, "grounded_ids": True}
    if context:
        answer, removed = remove_unshown_ids(answer, context)
        grounding["removed_ids"] = removed
        answer = enforce_ids(answer, context, check_grounding(answer, context))
    t_gen = time.perf_counter()

    yield "done", {
        "query": query, "parsed": pq.to_dict(), "answer": answer, "products": cards,
        "retrieved_ids": [p["id"] for p in retrieved], "no_match": not context,
        "unmet_constraints": unmet, "grounding": grounding,
        "best_similarity": getattr(pipeline.retriever, "last_best_similarity", None),
        "llm": getattr(gen, "model_id", None), "generation": {},
        "latency_ms": {"parse": round((t_parse - t_start) * 1000, 1),
                       "retrieve": round((t_retr - t_parse) * 1000, 1),
                       "generate": round((t_gen - t_retr) * 1000, 1),
                       "ttft": round(ttft * 1000, 1) if ttft is not None else None,
                       "total": round((t_gen - t_start) * 1000, 1)},
    }

### 📘 2.9 `src/api/main.py`

**Purpose:** loads Whisper at startup (after the pipeline; a failure leaves the text API up), adds `/transcribe`, `/recommend/audio`, `/speak`

**Expected Results:** `Writing /content/ShopTalk/src/api/main.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/api/main.py
"""FastAPI routing engine for the ShopTalk Inference API (Phase 7, voice added in Phase 10).

This application exposes endpoints for grounded catalog query recommendations,
Server-Sent Events streaming, explicit and implicit feedback logging, and system telemetry.

Phase 10 (voice): speech-to-text runs in front of the unchanged text pipeline.
  POST /transcribe        audio file -> transcript (the UI shows it so the user can fix mistakes)
  POST /recommend/audio   audio file -> transcript -> same pipeline as /recommend
  POST /speak             text -> WAV (optional text-to-speech, only when SHOPTALK_TTS=1)
Environment: SHOPTALK_ASR=0 disables speech-to-text; SHOPTALK_ASR_MODEL overrides rag_config["asr"]["model_id"].

To start manually:
    uvicorn src.api.main:app --host 0.0.0.0 --port 8000
"""
from __future__ import annotations

import json
import os
import re
import threading
import time
import traceback
import uuid
from contextlib import asynccontextmanager
from typing import Callable, Optional

from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile
from fastapi.concurrency import run_in_threadpool
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, RedirectResponse, Response, StreamingResponse

from src.asr.audio import AudioError, decode_audio
from src.asr.text_norm import transcript_to_query

from .schemas import (AudioRecommendResponse, FeedbackRequest, HealthResponse, HistoryTurn, RecommendRequest,
                      RecommendResponse, SpeakRequest, TranscribeResponse)
from .service import JsonlLogger, Metrics, log_record, run_stream, sse

API_VERSION = "0.10.0"
MAX_HISTORY = 3
MAX_AUDIO_BYTES = 10 * 1024 * 1024   # 10 MB upload limit
MAX_AUDIO_S = 30.0                   # Whisper's window; spoken shopping queries are a few seconds
_PID_RE = re.compile(r"^[A-Za-z0-9_\-]{1,40}$")


def _default_pipeline_factory() -> Callable[[], object]:
    """
    Factory function resolving which pipeline (real GPU vs mock fake) to construct.

    Returns:
        Callable: Constructor lambda that builds the active execution pipeline.
    """
    def build():
        if os.environ.get("SHOPTALK_FAKE") == "1":
            from .fake_pipeline import build_fake_pipeline
            return build_fake_pipeline()
        from src.rag.factory import build_pipeline
        repo = os.environ.get("SHOPTALK_REPO_DIR", os.getcwd())
        config = os.environ.get("SHOPTALK_CONFIG", os.path.join(repo, "src/rag/rag_config.json"))
        local = os.environ.get("SHOPTALK_LOCAL_CHROMA", "/tmp/shoptalk_chroma")

        # Build and return the real GPU-based RAG pipeline
        pipeline = build_pipeline(config, load_llm=True, repo_dir=repo, local_chroma_dir=local)
        import json
        pipeline.embed_model_name = json.load(open(config)).get("embed_model")
        return pipeline
    return build


def create_app(pipeline_factory: Optional[Callable[[], object]] = None, log_dir: Optional[str] = None,
               load_in_background: bool = True) -> FastAPI:
    """
    Instantiates and configures the FastAPI web application registry.

    Args:
        pipeline_factory (Callable, optional): Custom builder callback.
        log_dir (str, optional): Target directory for JSONLines logs.
        load_in_background (bool): If True, model loading runs asynchronously.

    Returns:
        FastAPI: Registered web application instance.
    """
    factory = pipeline_factory or _default_pipeline_factory()
    repo = os.environ.get("SHOPTALK_REPO_DIR", os.getcwd())
    state = {"pipeline": None, "status": "loading", "error": None, "started": time.time(),
             "asr": None, "asr_status": "loading", "asr_error": None,
             "tts": None, "tts_status": "disabled", "tts_error": None}
    gpu_lock = threading.Lock() # One GPU context: lock restricts model inference to 1 concurrency
    logger = JsonlLogger(log_dir or os.environ.get("SHOPTALK_LOG_DIR", os.path.join(repo, "logs")))
    metrics = Metrics()

    def load_voice():
        """Phase 10: speech-to-text (loaded once, on the GPU) and optional text-to-speech (CPU).
        A failure here never takes the text API down: /health reports it in asr_status / tts_status."""
        if os.environ.get("SHOPTALK_ASR", "1") == "0":
            state["asr_status"] = "disabled"
        else:
            t0 = time.time()
            try:
                from src.asr.whisper_asr import build_asr
                cfg_path = os.environ.get("SHOPTALK_CONFIG", os.path.join(repo, "src/rag/rag_config.json"))
                cfg = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}
                state["asr"] = build_asr(cfg)
                state["asr_status"] = "ok"
                logger.write("events", {"event": "asr_loaded", "model": state["asr"].model_id,
                                        "load_s": round(time.time() - t0, 1)})
            except Exception as e:
                state["asr_status"], state["asr_error"] = "error", f"{type(e).__name__}: {e}"
                logger.write("events", {"event": "asr_failed", "error": state["asr_error"],
                                        "trace": traceback.format_exc()[-2000:]})
        from src.tts.mms_tts import build_tts, tts_enabled
        if tts_enabled():
            state["tts_status"] = "loading"
            try:
                state["tts"] = build_tts()
                state["tts_status"] = "ok"
                logger.write("events", {"event": "tts_loaded", "model": state["tts"].model_id})
            except Exception as e:
                state["tts_status"], state["tts_error"] = "error", f"{type(e).__name__}: {e}"
                logger.write("events", {"event": "tts_failed", "error": state["tts_error"],
                                        "trace": traceback.format_exc()[-2000:]})

    def load():
        """Worker function executed at startup to initialize the pipeline components."""
        t0 = time.time()
        try:
            pipeline = factory()
            load_voice()          # before status=ok, so /health "ok" means every model is loaded
            state["pipeline"] = pipeline
            state["status"] = "ok"
            logger.write("events", {"event": "startup", "load_s": round(time.time() - t0, 1),
                                    "llm": getattr(state["pipeline"].generator, "model_id", None),
                                    "asr": getattr(state["asr"], "model_id", None)})
        except Exception as e:
            state["status"], state["error"] = "error", f"{type(e).__name__}: {e}"
            logger.write("events", {"event": "startup_failed", "error": state["error"],
                                    "trace": traceback.format_exc()[-2000:]})

    @asynccontextmanager
    async def lifespan(app: FastAPI):
        """Asynchronous lifecycle hook controlling startup loads and shutdowns."""
        if load_in_background:
            threading.Thread(target=load, daemon=True).start()
        else:
            load()
        yield

    app = FastAPI(title="ShopTalk API", version=API_VERSION, lifespan=lifespan,
                  description="Grounded product recommendations from the ABO catalog (RAG).")
    origins = [o.strip() for o in os.environ.get("SHOPTALK_CORS_ORIGINS", "*").split(",")]
    app.add_middleware(CORSMiddleware, allow_origins=origins, allow_methods=["*"], allow_headers=["*"])
    app.state.shoptalk = state
    app.state.metrics = metrics
    app.state.logger = logger

    def ready_pipeline():
        """Asserts model components are fully ready; raises 503 HTTP errors if still loading."""
        if state["status"] == "loading":
            raise HTTPException(503, "Models are still loading; retry in a few seconds (see GET /health).")
        if state["status"] == "error":
            raise HTTPException(503, f"Pipeline failed to load: {state['error']}")
        return state["pipeline"]

    def history_dicts(req: RecommendRequest) -> list:
        """Cuts history stack list down to the maximum supported conversational turns."""
        return [h.model_dump() for h in req.history[-MAX_HISTORY:]]

    def ready_asr():
        """Speech-to-text model, or 503 while loading / when it failed or is disabled."""
        ready_pipeline()
        if state["asr_status"] != "ok":
            detail = {"disabled": "Speech-to-text is disabled on this server (SHOPTALK_ASR=0).",
                      "error": f"Speech-to-text failed to load: {state['asr_error']}"}.get(
                state["asr_status"], "Speech-to-text is still loading; retry in a few seconds.")
            raise HTTPException(503, detail)
        return state["asr"]

    async def read_audio(audio: UploadFile) -> bytes:
        data = await audio.read()
        if not data:
            raise HTTPException(400, "Empty audio file.")
        if len(data) > MAX_AUDIO_BYTES:
            raise HTTPException(413, f"Audio file is larger than {MAX_AUDIO_BYTES // (1024 * 1024)} MB.")
        return data

    def decode_or_400(data: bytes):
        """Audio -> 16 kHz mono (CPU, outside the GPU lock). Returns (waveform, seconds, decode_ms)."""
        t0 = time.perf_counter()
        try:
            wav, seconds = decode_audio(data, max_seconds=MAX_AUDIO_S)
        except AudioError as e:
            raise HTTPException(400, f"Could not use this audio: {e}")
        return wav, seconds, round((time.perf_counter() - t0) * 1000, 1)

    # ---------------- endpoints ----------------
    @app.get("/health", response_model=HealthResponse)
    def health():
        """Reports application loading status, model IDs, active collection sizes, and uptimes."""
        p = state["pipeline"]
        index_size, embed = None, None
        if p is not None:
            try:
                index_size = p.retriever.collection.count()
            except Exception:
                index_size = getattr(p.retriever, "size", None)
            embed = getattr(p, "embed_model_name", None)
        return HealthResponse(status=state["status"], llm=getattr(getattr(p, "generator", None), "model_id", None),
                              embed_model=embed, index_size=index_size,
                              uptime_s=round(time.time() - state["started"], 1), error=state["error"],
                              asr=getattr(state["asr"], "model_id", None),
                              asr_status=state["asr_status"] if state["asr_status"] != "error"
                              else f"error: {state['asr_error']}",
                              tts=getattr(state["tts"], "model_id", None),
                              tts_status=state["tts_status"] if state["tts_status"] != "error"
                              else f"error: {state['tts_error']}")

    @app.post("/recommend", response_model=RecommendResponse)
    async def recommend(req: RecommendRequest):
        """Synchronous recommendation endpoint matching, generating, and logging query solutions."""
        pipeline = ready_pipeline()
        rid = uuid.uuid4().hex[:16]
        t_arrive = time.perf_counter()

        def work():
            with gpu_lock:
                t_lock = time.perf_counter()
                result = pipeline.run(req.query, history=history_dicts(req))
            result["latency_ms"]["queue"] = round((t_lock - t_arrive) * 1000, 1)
            return result

        try:
            result = await run_in_threadpool(work)
        except Exception as e:
            metrics.inc("errors")
            logger.write("errors", {"request_id": rid, "query": req.query, "error": f"{type(e).__name__}: {e}",
                                    "trace": traceback.format_exc()[-2000:]})
            raise HTTPException(500, f"Internal error (request_id={rid})")

        metrics.inc("requests")
        metrics.inc("no_match", int(bool(result["no_match"])))
        metrics.observe(result["latency_ms"])
        logger.write("requests", log_record(rid, req.session_id, "/recommend", result))
        return RecommendResponse(
            request_id=rid, query=req.query, answer=result["answer"], products=result["products"],
            no_match=result["no_match"], unmet_constraints=result.get("unmet_constraints", []),
            parsed=result["parsed"], grounding=result["grounding"], llm=result.get("llm"),
            latency_ms=result["latency_ms"],
            history_entry={"parsed": result["parsed"], "products": result["products"]})

    @app.post("/recommend/stream")
    def recommend_stream(req: RecommendRequest):
        """Server-Sent Events streaming endpoint delivering metadata and tokens step-by-step."""
        pipeline = ready_pipeline()
        rid = uuid.uuid4().hex[:16]
        t_arrive = time.perf_counter()

        def events():
            gpu_lock.acquire()
            try:
                queue_ms = round((time.perf_counter() - t_arrive) * 1000, 1)
                for kind, data in run_stream(pipeline, req.query, history_dicts(req)):
                    if kind == "meta":
                        yield sse("meta", {"request_id": rid, **data})
                    elif kind == "token":
                        yield sse("token", data)
                    else:
                        data["latency_ms"]["queue"] = queue_ms
                        metrics.inc("stream_requests")
                        metrics.inc("no_match", int(bool(data["no_match"])))
                        metrics.observe({k: v for k, v in data["latency_ms"].items() if v is not None})
                        logger.write("requests", log_record(rid, req.session_id, "/recommend/stream", data))
                        yield sse("done", {"request_id": rid, **{k: data[k] for k in (
                            "query", "answer", "products", "no_match", "unmet_constraints", "parsed",
                            "grounding", "llm", "latency_ms")},
                            "history_entry": {"parsed": data["parsed"], "products": data["products"]}})
            except Exception as e:
                metrics.inc("errors")
                logger.write("errors", {"request_id": rid, "query": req.query, "error": f"{type(e).__name__}: {e}",
                                        "trace": traceback.format_exc()[-2000:]})
                yield sse("error", {"request_id": rid, "detail": "Internal error"})
            finally:
                gpu_lock.release()

        return StreamingResponse(events(), media_type="text/event-stream",
                                 headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

    # ---------------- Phase 10: voice ----------------
    @app.post("/transcribe", response_model=TranscribeResponse)
    async def transcribe(audio: UploadFile = File(..., description="Recording: wav, webm, ogg, mp3, m4a"),
                         session_id: Optional[str] = Form(None)):
        """Speech -> query text. The UI shows the transcript so the user can correct it before searching."""
        asr = ready_asr()
        data = await read_audio(audio)
        rid = uuid.uuid4().hex[:16]
        wav, seconds, decode_ms = decode_or_400(data)
        t_arrive = time.perf_counter()        # queue = waiting for the GPU lock, after decoding

        def work():
            with gpu_lock:
                t_lock = time.perf_counter()
                out = asr.transcribe(wav)
            out["queue_ms"] = round((t_lock - t_arrive) * 1000, 1)
            return out

        try:
            out = await run_in_threadpool(work)
        except Exception as e:
            metrics.inc("errors")
            logger.write("errors", {"request_id": rid, "endpoint": "/transcribe", "error": f"{type(e).__name__}: {e}",
                                    "trace": traceback.format_exc()[-2000:]})
            raise HTTPException(500, f"Internal error (request_id={rid})")
        latency = {"decode": decode_ms, "queue": out["queue_ms"], "asr": out["ms"]}
        query = transcript_to_query(out["text"])
        metrics.inc("transcribe")
        metrics.observe({"asr": out["ms"]})
        logger.write("asr", {"request_id": rid, "session_id": session_id, "endpoint": "/transcribe",
                             "text": query, "raw_text": out["text"], "model": out["model"],
                             "audio_s": round(seconds, 2), "bytes": len(data), "content_type": audio.content_type,
                             "latency_ms": latency})
        return TranscribeResponse(request_id=rid, text=query, raw_text=out["text"], model=out["model"],
                                  audio_s=round(seconds, 2), latency_ms=latency)

    @app.post("/recommend/audio", response_model=AudioRecommendResponse)
    async def recommend_audio(audio: UploadFile = File(...), session_id: Optional[str] = Form(None),
                              history: Optional[str] = Form(None, description="JSON list of earlier history_entry")):
        """One call for voice clients: speech-to-text, then exactly the /recommend pipeline on the transcript."""
        asr = ready_asr()
        pipeline = ready_pipeline()
        data = await read_audio(audio)
        try:
            hist = [HistoryTurn(**h).model_dump() for h in (json.loads(history) if history else [])][-MAX_HISTORY:]
        except (ValueError, TypeError) as e:
            raise HTTPException(422, f"history must be a JSON list of history entries: {e}")
        rid = uuid.uuid4().hex[:16]
        wav, seconds, decode_ms = decode_or_400(data)
        t_arrive = time.perf_counter()        # queue = waiting for the GPU lock, after decoding

        def work():
            with gpu_lock:
                t_lock = time.perf_counter()
                heard = asr.transcribe(wav)
                text = transcript_to_query(heard["text"]).strip()
                result = pipeline.run(text, history=hist) if text else None
            return heard, result, round((t_lock - t_arrive) * 1000, 1)

        try:
            heard, result, queue_ms = await run_in_threadpool(work)
        except Exception as e:
            metrics.inc("errors")
            logger.write("errors", {"request_id": rid, "endpoint": "/recommend/audio",
                                    "error": f"{type(e).__name__}: {e}", "trace": traceback.format_exc()[-2000:]})
            raise HTTPException(500, f"Internal error (request_id={rid})")
        if result is None:
            logger.write("asr", {"request_id": rid, "session_id": session_id, "endpoint": "/recommend/audio",
                                 "text": "", "model": heard["model"], "audio_s": round(seconds, 2)})
            raise HTTPException(422, "No speech was recognised in the recording.")
        lat = result["latency_ms"]
        lat.update(queue=queue_ms, decode=decode_ms, asr=heard["ms"],
                   total_with_asr=round(lat["total"] + heard["ms"] + decode_ms, 1))
        metrics.inc("requests")
        metrics.inc("audio_requests")
        metrics.inc("no_match", int(bool(result["no_match"])))
        metrics.observe(lat)
        rec = log_record(rid, session_id, "/recommend/audio", result)
        rec.update(input="voice", transcript=result["query"], raw_transcript=heard["text"], asr_model=heard["model"],
                   audio_s=round(seconds, 2))
        logger.write("requests", rec)
        return AudioRecommendResponse(
            request_id=rid, query=result["query"], answer=result["answer"], products=result["products"],
            no_match=result["no_match"], unmet_constraints=result.get("unmet_constraints", []),
            parsed=result["parsed"], grounding=result["grounding"], llm=result.get("llm"), latency_ms=lat,
            history_entry={"parsed": result["parsed"], "products": result["products"]},
            transcript=result["query"], raw_transcript=heard["text"], asr_model=heard["model"],
            audio_s=round(seconds, 2))

    @app.post("/speak")
    def speak(req: SpeakRequest):
        """Optional text-to-speech (SHOPTALK_TTS=1): returns audio/wav of the answer, IDs removed."""
        if state["tts_status"] == "disabled":
            raise HTTPException(404, "Text-to-speech is disabled on this server (set SHOPTALK_TTS=1).")
        if state["tts_status"] != "ok":
            raise HTTPException(503, f"Text-to-speech is not ready: {state['tts_status']} {state['tts_error'] or ''}")
        try:
            out = state["tts"].synthesize(req.text)
        except ValueError as e:
            raise HTTPException(400, str(e))
        metrics.inc("tts")
        metrics.observe({"tts": out["ms"]})
        return Response(content=out["wav"], media_type="audio/wav",
                        headers={"X-TTS-ms": str(out["ms"]), "X-Audio-Seconds": str(out["audio_s"])})

    @app.post("/feedback")
    def feedback(fb: FeedbackRequest):
        """Logs explicit user thumbs-up/down ratings and comments into telemetry."""
        metrics.inc("feedback_up" if fb.rating == "up" else "feedback_down")
        logger.write("feedback", {"type": "explicit", **fb.model_dump()})
        return {"status": "recorded"}

    @app.get("/click/{request_id}/{product_id}")
    def click(request_id: str, product_id: str, rank: Optional[int] = None, session_id: Optional[str] = None):
        """Registers implicit product links click feedback, redirecting standard requests to the store link."""
        if not _PID_RE.match(product_id) or not _PID_RE.match(request_id):
            raise HTTPException(400, "Invalid id")
        metrics.inc("clicks")
        logger.write("feedback", {"type": "click", "request_id": request_id, "product_id": product_id,
                                  "rank": rank, "session_id": session_id})
        template = getattr(state["pipeline"], "link_template", None) or "https://www.amazon.com/dp/{id}"
        return RedirectResponse(template.format(id=product_id), status_code=307)

    @app.get("/metrics")
    def get_metrics():
        """Returns in-memory requests counters and rolling percentiles timing statistics."""
        return metrics.snapshot()

    @app.exception_handler(Exception)
    async def unhandled(request: Request, exc: Exception):
        """Catches and drops unhandled system failures into clean 500 responses."""
        return JSONResponse(status_code=500, content={"detail": "Internal error"})

    return app


app = create_app()

### 📘 2.10 `src/ui/client.py`

**Purpose:** `transcribe()` and `speak()` client methods

**Expected Results:** `Writing /content/ShopTalk/src/ui/client.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/ui/client.py
"""HTTP client for the ShopTalk inference API (Phase 7), used by the Phase 8 UI.

Keeps the UI free of HTTP details: the Gradio handlers call these methods and get plain dicts back.
Only `requests` is needed, so the same client works in Colab, in tests and later on AWS (Phase 11).

Endpoints used:
  GET  /health                          -> status banner
  POST /recommend/stream                -> SSE events: meta (products), token (answer text), done, error
  POST /recommend                       -> non-streaming fallback
  POST /feedback                        -> thumbs up / down
  GET  /click/{request_id}/{product_id} -> logs the click, then 307-redirects to the product page
  POST /transcribe                      -> speech-to-text for the microphone input (Phase 10)
  POST /speak                           -> optional text-to-speech of the answer (Phase 10)
"""
from __future__ import annotations

import json
import mimetypes
import os
from typing import Iterator, Optional

import requests


class ApiError(RuntimeError):
    """Raised when the API is unreachable or answers with an error the user should see."""


class ShopTalkClient:
    """
    Thin wrapper around the Phase 7 endpoints.

    Args:
        base_url (str): API address, e.g. "http://127.0.0.1:8000" or a public tunnel URL.
        timeout (float): Seconds to wait for a full response (streaming reads use it per chunk).
    """

    def __init__(self, base_url: str, timeout: float = 300.0):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.session = requests.Session()

    # ---------------- helpers ----------------
    def _url(self, path: str) -> str:
        return f"{self.base_url}{path}"

    @staticmethod
    def _raise_for(r: requests.Response) -> None:
        """Turns HTTP errors into ApiError with the API's own message (e.g. 503 while loading)."""
        if r.status_code < 400:
            return
        try:
            detail = r.json().get("detail", r.text)
        except ValueError:
            detail = r.text
        raise ApiError(f"API error {r.status_code}: {detail}")

    # ---------------- endpoints ----------------
    def health(self) -> dict:
        """Returns /health, or {'status': 'unreachable', 'error': ...} if the API cannot be reached."""
        try:
            r = self.session.get(self._url("/health"), timeout=10)
            self._raise_for(r)
            return r.json()
        except (requests.RequestException, ApiError) as e:
            return {"status": "unreachable", "error": str(e)}

    def recommend(self, query: str, history: Optional[list] = None, session_id: Optional[str] = None) -> dict:
        """Non-streaming call to POST /recommend."""
        try:
            r = self.session.post(self._url("/recommend"), timeout=self.timeout,
                                  json={"query": query, "history": history or [], "session_id": session_id})
        except requests.RequestException as e:
            raise ApiError(f"API unreachable: {e}") from e
        self._raise_for(r)
        return r.json()

    def stream(self, query: str, history: Optional[list] = None,
               session_id: Optional[str] = None) -> Iterator[tuple]:
        """
        Calls POST /recommend/stream and yields (event, data) as each Server-Sent Event arrives.

        Events: ("meta", {request_id, products, ...}), ("token", {text}), ("done", {...full response}),
        ("error", {request_id, detail}).
        """
        try:
            r = self.session.post(self._url("/recommend/stream"), stream=True, timeout=(10, self.timeout),
                                  json={"query": query, "history": history or [], "session_id": session_id})
        except requests.RequestException as e:
            raise ApiError(f"API unreachable: {e}") from e
        with r:
            self._raise_for(r)
            yield from parse_sse(r.iter_content(chunk_size=None))

    def feedback(self, request_id: str, rating: str, comment: Optional[str] = None,
                 session_id: Optional[str] = None) -> dict:
        """POST /feedback with rating 'up' or 'down'."""
        try:
            r = self.session.post(self._url("/feedback"), timeout=10, json={
                "request_id": request_id, "rating": rating, "comment": comment or None, "session_id": session_id})
        except requests.RequestException as e:
            raise ApiError(f"API unreachable: {e}") from e
        self._raise_for(r)
        return r.json()

    def transcribe(self, audio_path: str, session_id: Optional[str] = None) -> dict:
        """POST /transcribe with a recorded file. Returns {request_id, text, model, audio_s, latency_ms}."""
        mime = mimetypes.guess_type(audio_path)[0] or "application/octet-stream"
        try:
            with open(audio_path, "rb") as f:
                r = self.session.post(self._url("/transcribe"), timeout=60,
                                      files={"audio": (os.path.basename(audio_path), f, mime)},
                                      data={"session_id": session_id} if session_id else {})
        except requests.RequestException as e:
            raise ApiError(f"API unreachable: {e}") from e
        self._raise_for(r)
        return r.json()

    def speak(self, text: str) -> bytes:
        """POST /speak. Returns WAV bytes; raises ApiError (e.g. 404 when TTS is disabled on the server)."""
        try:
            r = self.session.post(self._url("/speak"), json={"text": text[:2000]}, timeout=60)
        except requests.RequestException as e:
            raise ApiError(f"API unreachable: {e}") from e
        self._raise_for(r)
        return r.content

    def click(self, request_id: str, product_id: str, rank: Optional[int] = None,
              session_id: Optional[str] = None) -> Optional[str]:
        """
        Logs a click via GET /click/... without following the redirect.

        Returns:
            str | None: The product page URL from the redirect, or None if the API did not redirect.
        """
        params = {k: v for k, v in (("rank", rank), ("session_id", session_id)) if v is not None}
        try:
            r = self.session.get(self._url(f"/click/{request_id}/{product_id}"), params=params,
                                 allow_redirects=False, timeout=10)
        except requests.RequestException:
            return None
        return r.headers.get("location") if r.status_code in (301, 302, 303, 307, 308) else None


def parse_sse(chunks) -> Iterator[tuple]:
    """
    Parses a Server-Sent Events byte stream into (event, data) tuples.

    Works on raw chunks (not lines) so tokens are shown as soon as they arrive; requests' iter_lines
    would buffer them. Events are separated by a blank line; `data:` is JSON.
    """
    buf = ""
    for chunk in chunks:
        if not chunk:
            continue
        buf += chunk.decode("utf-8", errors="replace") if isinstance(chunk, bytes) else chunk
        buf = buf.replace("\r\n", "\n")
        while "\n\n" in buf:
            block, buf = buf.split("\n\n", 1)
            event, data_lines = "message", []
            for line in block.split("\n"):
                if line.startswith("event:"):
                    event = line[6:].strip()
                elif line.startswith("data:"):
                    data_lines.append(line[5:].lstrip())
            if data_lines:
                raw = "\n".join(data_lines)
                try:
                    yield event, json.loads(raw)
                except ValueError:
                    yield event, {"text": raw}

### 📘 2.11 `src/ui/render.py`

**Purpose:** status banner shows the speech-to-text model and read-aloud availability

**Expected Results:** `Writing /content/ShopTalk/src/ui/render.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/ui/render.py
"""HTML / Markdown rendering for the Phase 8 UI. Pure functions (no Gradio), so they are unit-tested.

Everything that comes from the catalog or the LLM is HTML-escaped before it is shown.
"""
from __future__ import annotations

import html
import re
from typing import Optional
from urllib.parse import quote

ID_RE = re.compile(r"\[ID:\s*([A-Za-z0-9_\-]+)\]")

CSS = """
.st-grid {display:grid; grid-template-columns:repeat(auto-fill,minmax(180px,1fr)); gap:12px;}
.st-card {border:1px solid var(--border-color-primary); border-radius:10px; padding:10px;
          background:var(--background-fill-secondary); display:flex; flex-direction:column; gap:6px;}
.st-img {width:100%; aspect-ratio:1/1; object-fit:contain; border-radius:6px; background:#fff;}
.st-noimg {width:100%; aspect-ratio:1/1; border-radius:6px; display:flex; align-items:center;
           justify-content:center; font-size:42px; background:var(--background-fill-primary);}
.st-title {font-weight:600; font-size:0.92em; line-height:1.25; display:-webkit-box;
           -webkit-line-clamp:3; -webkit-box-orient:vertical; overflow:hidden;}
.st-meta {font-size:0.8em; opacity:0.8;}
.st-price {font-weight:700;}
.st-id {font-family:monospace; font-size:0.78em; opacity:0.75;}
.st-chip {display:inline-block; font-size:0.72em; padding:1px 6px; border-radius:8px; margin-right:4px;
          background:var(--color-accent-soft);}
.st-link {margin-top:auto; font-size:0.85em; font-weight:600;}
.st-note {font-size:0.85em; padding:8px 10px; border-radius:8px; background:var(--background-fill-secondary);
          margin-bottom:10px;}
"""

EMPTY_PRODUCTS = "<div class='st-note'>Product recommendations will appear here.</div>"


def esc(x) -> str:
    return html.escape("" if x is None else str(x), quote=True)


def click_href(link_base: str, request_id: Optional[str], product_id: str, rank: int,
               session_id: Optional[str] = None, fallback: Optional[str] = None) -> str:
    """
    Link used on a product card. It points at the UI's /go route, which logs the click through the
    API's /click endpoint and then redirects to the product page. Without a request_id there is
    nothing to log against, so the direct product link is used.
    """
    if not request_id:
        return fallback or "#"
    href = f"{link_base.rstrip('/')}/go/{quote(request_id)}/{quote(product_id)}?rank={rank}"
    if session_id:
        href += f"&session_id={quote(session_id)}"
    return href


def format_price(p: dict) -> str:
    if p.get("price") is None or p.get("price_unknown"):
        return "price n/a"
    return f"${p['price']:,.2f}"


def product_cards_html(products: list, request_id: Optional[str] = None, link_base: str = "",
                       session_id: Optional[str] = None, unmet: Optional[list] = None,
                       no_match: bool = False) -> str:
    """Renders the product grid shown next to the chat."""
    if no_match or not products:
        return ("<div class='st-note'>No matching products in the catalog. Try a broader description "
                "or relax a filter such as price or color.</div>")
    parts = []
    if unmet:
        parts.append(f"<div class='st-note'>⚠️ None of these has everything you asked for "
                     f"(not matched: {esc(', '.join(unmet))}). These are the closest options.</div>")
    parts.append("<div class='st-grid'>")
    for rank, p in enumerate(products, 1):
        pid = str(p.get("id", ""))
        img = p.get("image_url")
        img_html = (f"<img class='st-img' src='{esc(img)}' alt='{esc(p.get('title'))}' loading='lazy'>"
                    if isinstance(img, str) and img.startswith(("http://", "https://"))
                    else "<div class='st-noimg'>🛍️</div>")
        meta = " · ".join(esc(x) for x in (p.get("brand"), p.get("color"), p.get("product_type")) if x)
        chips = "".join(f"<span class='st-chip'>✓ {esc(k)}</span>"
                        for k, v in (p.get("matched") or {}).items() if v)
        href = click_href(link_base, request_id, pid, rank, session_id, fallback=p.get("link"))
        parts.append(
            f"<div class='st-card'>{img_html}"
            f"<div class='st-title'>{rank}. {esc(p.get('title'))}</div>"
            f"<div class='st-meta'>{meta}</div>"
            f"<div><span class='st-price'>{esc(format_price(p))}</span></div>"
            f"<div>{chips}</div>"
            f"<div class='st-id'>ID: {esc(pid)}</div>"
            f"<a class='st-link' href='{esc(href)}' target='_blank' rel='noopener'>View product ↗</a>"
            f"</div>")
    parts.append("</div>")
    return "".join(parts)


def format_answer(text: str) -> str:
    """Chat answer as Markdown: product citations like [ID: B082SP22KL] are shown in bold code style.
    Angle brackets are escaped so catalog titles or LLM output cannot inject HTML into the chat."""
    text = (text or "").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    return ID_RE.sub(lambda m: f"**`{m.group(1)}`**", text).strip()


def latency_line(latency_ms: Optional[dict], client_ms: Optional[dict] = None) -> str:
    """Small status line under the chat: server-side stage times and what the user actually waited."""
    if not latency_ms:
        return ""
    keys = [k for k in ("parse", "retrieve", "generate", "queue", "total") if latency_ms.get(k) is not None]
    s = " · ".join(f"{k} {latency_ms[k]:,.0f} ms" for k in keys)
    if client_ms:
        s += " | UI: " + " · ".join(f"{k.replace('_ms', '').replace('_', ' ')} {v:,.0f} ms"
                                    for k, v in client_ms.items() if v is not None)
    return f"⏱️ {s}"


def health_banner(h: dict, api_url: str) -> str:
    status = h.get("status")
    if status == "ok":
        voice = ""
        if h.get("asr"):
            voice = f" · 🎤 `{h.get('asr')}`"
        elif h.get("asr_status") and h.get("asr_status") != "ok":
            voice = f" · 🎤 voice input unavailable ({esc(h.get('asr_status'))})"
        if h.get("tts"):
            voice += " · 🔊 read-aloud available"
        return (f"🟢 **API ready** · LLM `{h.get('llm')}` · index {h.get('index_size') or 0:,} products"
                f"{voice} · `{api_url}`")
    if status == "loading":
        return f"🟡 **API is loading models** (1–3 min). Refresh the status in a moment · `{api_url}`"
    return f"🔴 **API not available** ({esc(h.get('error') or status)}) · `{api_url}`"

### 📘 2.12 `src/ui/app.py`

**Purpose:** 🎤 microphone next to the text box → transcript into the box (user checks, then Send); 🔊 read-aloud checkbox, off by default

**Expected Results:** `Writing /content/ShopTalk/src/ui/app.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/ui/app.py
"""ShopTalk chat UI (Phase 8): a Gradio app on top of the Phase 7 inference API.

What the user gets:
  * a chat box: the answer streams in token by token (POST /recommend/stream)
  * product cards next to the chat (image, title, brand, price, ID, "View product" link)
  * follow-ups ("something similar but cheaper"): each response's `history_entry` is kept and sent back
  * 👍 / 👎 (+ optional comment) on the latest answer -> POST /feedback
  * clicks on "View product" go through /go/... on this UI server, which logs them via the API's
    /click endpoint and then redirects to the product page
  * Phase 10: 🎤 microphone input -> POST /transcribe -> the transcript is put in the text box, so the user
    sees what was heard and can fix it before pressing Send; optional 🔊 "read answers aloud" (POST /speak),
    off by default

The UI holds no model: it only talks HTTP to the API, so the API can stay private (127.0.0.1) and
only the UI needs a public URL. The same file runs in Colab and in Docker / AWS (Phase 11).

Run:
    SHOPTALK_API_URL=http://127.0.0.1:8000 python -m src.ui.app        # serves on port 7860
"""
from __future__ import annotations

import os
import re
import tempfile
import time
import uuid
from typing import Optional

import gradio as gr
from fastapi import FastAPI
from fastapi.responses import RedirectResponse

from .client import ApiError, ShopTalkClient
from .render import (CSS, EMPTY_PRODUCTS, format_answer, health_banner, latency_line, product_cards_html)

MAX_HISTORY = 3                      # the API uses the last 3 turns
PRODUCT_URL = "https://www.amazon.com/dp/{id}"
_ID_RE = re.compile(r"^[A-Za-z0-9_\-]{1,64}$")

EXAMPLES = ["Show me a red shirt for men under $50", "Waterproof hiking boots, size 10",
            "A gift for someone who loves coffee", "Something cozy for a small living room",
            "Find me a black leather wallet"]


# ---------------- per-browser-session state ----------------
def new_state(session_id: Optional[str] = None) -> dict:
    """State kept per browser tab: session id, last turns for follow-ups, and the latest request."""
    return {"session_id": session_id or f"ui-{uuid.uuid4().hex[:12]}", "history": [],
            "last_request_id": None, "voted": {}}


def _assistant(chat: list, text: str) -> list:
    """Sets the text of the last assistant message (adds one if the last message is the user's)."""
    if chat and chat[-1].get("role") == "assistant":
        chat[-1] = {"role": "assistant", "content": text}
    else:
        chat.append({"role": "assistant", "content": text})
    return chat


# ---------------- handlers (plain functions, unit-tested without a browser) ----------------
def add_user_message(message: str, chat: Optional[list]):
    """Step 1 of a turn: show the user's message at once and clear the textbox."""
    chat = list(chat or [])
    message = (message or "").strip()
    if message:
        chat.append({"role": "user", "content": message})
    return "", chat, message


def chat_turn(client: ShopTalkClient, message: str, chat: Optional[list], state: Optional[dict],
              link_base: str = "", use_stream: bool = True):
    """
    Step 2 of a turn: call the API and stream the result into the UI.

    Yields (chat, products_html, latency_markdown, state) every time something new arrives:
    first the product cards (meta event), then the growing answer (token events), then the final
    answer (done event; the API cleans citations, so it can differ slightly from the streamed text).
    """
    chat = list(chat or [])
    state = dict(state or new_state())
    if not message:
        yield chat, gr.update(), gr.update(), state
        return
    if len(message) > 500:
        yield _assistant(chat, "That message is too long (500 characters max). Please shorten it."), \
            gr.update(), "", state
        return

    sid, history = state["session_id"], state["history"][-MAX_HISTORY:]
    t0 = time.perf_counter()
    products_html, text = gr.update(), ""
    client_ms = {"products_ms": None, "first_token_ms": None, "total_ms": None}
    done = None
    try:
        if use_stream:
            yield _assistant(chat, "🔎 Searching the catalog…"), gr.update(), "", state
            for event, data in client.stream(message, history=history, session_id=sid):
                if event == "meta":
                    client_ms["products_ms"] = (time.perf_counter() - t0) * 1000
                    state["last_request_id"] = data.get("request_id")
                    products_html = product_cards_html(
                        data.get("products", []), data.get("request_id"), link_base, sid,
                        data.get("unmet_constraints"), data.get("no_match", False))
                    yield _assistant(chat, "✍️ Writing an answer…"), products_html, "", state
                elif event == "token":
                    if client_ms["first_token_ms"] is None:
                        client_ms["first_token_ms"] = (time.perf_counter() - t0) * 1000
                    text += data.get("text", "")
                    yield _assistant(chat, format_answer(text) + " ▌"), products_html, "", state
                elif event == "done":
                    done = data
                elif event == "error":
                    raise ApiError(f"{data.get('detail', 'error')} (request_id={data.get('request_id')})")
        else:
            yield _assistant(chat, "🔎 Searching and writing an answer…"), gr.update(), "", state
            done = client.recommend(message, history=history, session_id=sid)
    except ApiError as e:
        msg = str(e)
        if "503" in msg:
            msg = "The models are still loading on the server. Please try again in a minute."
        yield _assistant(chat, f"⚠️ {msg}"), products_html, "", state
        return

    if done is None:
        yield _assistant(chat, "⚠️ The answer was cut off. Please try again."), products_html, "", state
        return

    client_ms["total_ms"] = (time.perf_counter() - t0) * 1000
    rid = done.get("request_id")
    state["last_request_id"] = rid
    state["history"] = (state["history"] + [done["history_entry"]])[-MAX_HISTORY:]
    products_html = product_cards_html(done.get("products", []), rid, link_base, sid,
                                       done.get("unmet_constraints"), done.get("no_match", False))
    state["last_response"] = {"request_id": rid, "products": done.get("products", []),
                              "no_match": done.get("no_match"), "latency_ms": done.get("latency_ms"),
                              "client_ms": {k: round(v) for k, v in client_ms.items() if v is not None},
                              "answer": done.get("answer", "")}
    yield (_assistant(chat, format_answer(done.get("answer", ""))), products_html,
           latency_line(done.get("latency_ms"), {k: v for k, v in client_ms.items() if v is not None}), state)


def send_feedback(client: ShopTalkClient, rating: str, comment: str, state: Optional[dict]):
    """👍 / 👎 on the latest answer. Returns (status_markdown, comment_box_value, state)."""
    state = dict(state or new_state())
    rid = state.get("last_request_id")
    if not rid:
        return "Ask something first, then rate the answer.", comment, state
    voted = dict(state.get("voted") or {})
    if voted.get(rid) == rating and not (comment or "").strip():
        return "Already recorded, thanks!", comment, state
    try:
        client.feedback(rid, rating, (comment or "").strip()[:1000] or None, state["session_id"])
    except ApiError as e:
        return f"⚠️ Could not send feedback: {e}", comment, state
    voted[rid] = rating
    state["voted"] = voted
    return ("Thanks! 👍 recorded." if rating == "up" else "Thanks, 👎 recorded. It will help us improve."), "", state


def transcribe_audio(client: ShopTalkClient, audio_path: Optional[str], state: Optional[dict]):
    """🎤 -> text box. Returns (textbox_update, heard_markdown). The query is NOT sent automatically:
    the user checks the transcript (brands, sizes and prices are easy to mishear) and presses Send."""
    if not audio_path:
        return gr.update(), ""
    sid = (state or {}).get("session_id")
    try:
        out = client.transcribe(audio_path, session_id=sid)
    except ApiError as e:
        msg = str(e)
        if "503" in msg:
            msg = "Speech-to-text is not available on the server right now; please type your request."
        elif "400" in msg:
            msg = "Could not understand that recording; please try again or type your request."
        return gr.update(), f"⚠️ {msg}"
    text = (out.get("text") or "").strip()
    if not text:
        return gr.update(), "🎤 I didn't catch any words; please try again."
    lat = out.get("latency_ms") or {}
    return text, (f"🎤 Heard: *{text}* — check it, edit if needed, then press **Send** "
                  f"(speech-to-text {lat.get('asr', 0):,.0f} ms)")


_ID_TAG = re.compile(r"\[\s*ID\s*:[^\]]*\]")


def speak_answer(client: ShopTalkClient, enabled: bool, state: Optional[dict]):
    """🔊 After a turn: read the latest answer aloud when the checkbox is on. Returns an Audio update."""
    answer = ((state or {}).get("last_response") or {}).get("answer", "")
    if not enabled or not answer:
        return gr.update(value=None, visible=False)
    try:
        wav = client.speak(_ID_TAG.sub("", answer))
    except ApiError:
        return gr.update(value=None, visible=False)
    f = tempfile.NamedTemporaryFile(prefix="shoptalk_tts_", suffix=".wav", delete=False)
    f.write(wav)
    f.close()
    return gr.update(value=f.name, visible=True)


def clear_chat(state: Optional[dict]):
    """New conversation: empties the chat and the follow-up history, keeps the session id."""
    s = new_state((state or {}).get("session_id"))
    return [], EMPTY_PRODUCTS, "", "", s


# ---------------- Gradio layout ----------------
def build_ui(client: ShopTalkClient, api_url: str, link_base: str = "", use_stream: bool = True) -> gr.Blocks:
    """Builds the Gradio Blocks app (not launched)."""
    with gr.Blocks(title="ShopTalk") as demo:
        gr.Markdown("# 🛍️ ShopTalk\nAsk for products in plain language. Follow-ups such as "
                    "*\"something similar but cheaper\"* use the previous results.")
        with gr.Row():
            status = gr.Markdown(health_banner(client.health(), api_url))
            refresh = gr.Button("Refresh status", size="sm", scale=0)
        state = gr.State(new_state)
        pending = gr.State("")
        with gr.Row(equal_height=False):
            with gr.Column(scale=5):
                chatbot = gr.Chatbot(label="Chat", height=520)
                with gr.Row():
                    msg = gr.Textbox(placeholder="e.g. Waterproof hiking boots, size 10", show_label=False,
                                     scale=5, max_lines=3, autofocus=True)
                    send = gr.Button("Send", variant="primary", scale=1)
                with gr.Row():
                    mic = gr.Audio(sources=["microphone"], type="filepath", format="wav",
                                   label="🎤 Or say it (the words appear in the box above)", scale=5)
                    speak_on = gr.Checkbox(value=False, label="🔊 Read answers aloud", scale=1)
                heard = gr.Markdown("")
                answer_audio = gr.Audio(label="Answer (audio)", autoplay=True, interactive=False, visible=False)
                gr.Examples(EXAMPLES, inputs=msg, label="Try")
                latency = gr.Markdown("")
                with gr.Row():
                    up = gr.Button("👍 Helpful", size="sm")
                    down = gr.Button("👎 Not helpful", size="sm")
                    clear = gr.Button("🧹 New chat", size="sm")
                comment = gr.Textbox(placeholder="Optional comment for your rating", show_label=False, max_lines=2)
                fb_status = gr.Markdown("")
            with gr.Column(scale=6):
                gr.Markdown("### Products")
                products = gr.HTML(EMPTY_PRODUCTS)

        def _turn(message, chat, st):
            yield from chat_turn(client, message, chat, st, link_base, use_stream)

        for trigger in (msg.submit, send.click):
            trigger(add_user_message, [msg, chatbot], [msg, chatbot, pending], queue=False) \
                .then(_turn, [pending, chatbot, state], [chatbot, products, latency, state]) \
                .then(lambda on, st: speak_answer(client, on, st), [speak_on, state], answer_audio)
        for rec_event in (mic.stop_recording, mic.upload):
            rec_event(lambda path, st: transcribe_audio(client, path, st), [mic, state], [msg, heard])
        up.click(lambda c, s: send_feedback(client, "up", c, s), [comment, state], [fb_status, comment, state])
        down.click(lambda c, s: send_feedback(client, "down", c, s), [comment, state], [fb_status, comment, state])
        clear.click(clear_chat, [state], [chatbot, products, latency, fb_status, state]) \
            .then(lambda: ("", gr.update(value=None, visible=False)), None, [heard, answer_audio])
        refresh.click(lambda: health_banner(client.health(), api_url), None, status)
        demo.load(lambda: health_banner(client.health(), api_url), None, status)
    demo.queue(default_concurrency_limit=8)
    return demo


def create_server(api_url: str, use_stream: bool = True) -> FastAPI:
    """
    FastAPI app that serves the Gradio UI at / and the click-tracking redirect at /go/....

    /go/{request_id}/{product_id}?rank=N logs the click through the API (server side, so the API does
    not need to be public) and redirects the browser to the product page.
    """
    client = ShopTalkClient(api_url)
    server = FastAPI(title="ShopTalk UI")

    @server.get("/go/{request_id}/{product_id}")
    def go(request_id: str, product_id: str, rank: Optional[int] = None, session_id: Optional[str] = None):
        if not (_ID_RE.match(request_id) and _ID_RE.match(product_id)):
            return RedirectResponse("/", status_code=307)
        target = client.click(request_id, product_id, rank=rank, session_id=session_id)
        return RedirectResponse(target or PRODUCT_URL.format(id=product_id), status_code=307)

    @server.get("/ui-health")
    def ui_health():
        return {"ui": "ok", "api": client.health().get("status"), "api_url": api_url}

    demo = build_ui(client, api_url, link_base="", use_stream=use_stream)
    return gr.mount_gradio_app(server, demo, path="/", css=CSS, theme=gr.themes.Soft())


def main():
    import uvicorn
    api_url = os.environ.get("SHOPTALK_API_URL", "http://127.0.0.1:8000")
    port = int(os.environ.get("SHOPTALK_UI_PORT", "7860"))
    use_stream = os.environ.get("SHOPTALK_UI_STREAM", "1") != "0"
    uvicorn.run(create_server(api_url, use_stream), host="0.0.0.0", port=port, log_level="info")


if __name__ == "__main__":
    main()

### 📘 2.13 Self-test in fake mode

**Purpose:** Start the real API with `SHOPTALK_FAKE=1` (no GPU models) on a spare port and call every voice endpoint, including error cases; then build the UI layout. Catches wiring errors before any model is loaded.

**Expected Results:** `fake-mode self-test ok` with transcribe / recommend-audio / speak / error codes printed.

**Parameters:** none

In [ ]:
from src.eval.api_server import ApiServer
from src.eval.client import ShopTalkClient as EvalClient
import requests
for m in [m for m in list(sys.modules) if m.startswith(("src.asr", "src.tts", "src.api", "src.ui"))]: del sys.modules[m]
from src.asr.audio import decode_audio, write_wav, to_wav_bytes, AudioError
from src.asr.text_norm import transcript_to_query, normalize, wer, corpus_wer, key_term_recall, filter_agreement, speakable

assert transcript_to_query("women's sandals under thirty dollars.") == "women's sandals under $30."
assert normalize("Size 9.5, $30") == normalize("size nine point five 30 dollars")
fake = ApiServer(REPO_LINK, API_APP, 8099, {"SHOPTALK_FAKE": "1", "SHOPTALK_TTS": "1", "SHOPTALK_LOG_DIR": str(LOG_DIR / "selftest"),
                 "SHOPTALK_FAKE_TRANSCRIPT": "red sneakers for men under thirty dollars"}, LOG_DIR / "selftest_uvicorn.log")
fake.start()
try:
    EvalClient(fake.url).wait_healthy(120, proc=fake.proc, verbose=False)
    wav = to_wav_bytes(0.1 * np.sin(np.arange(32000) / 5))
    t = requests.post(fake.url + "/transcribe", files={"audio": ("a.wav", wav, "audio/wav")}).json()
    r = requests.post(fake.url + "/recommend/audio", files={"audio": ("a.wav", wav, "audio/wav")}).json()
    s = requests.post(fake.url + "/speak", json={"text": "Try [ID: X] for $5"})
    bad = requests.post(fake.url + "/transcribe", files={"audio": ("a.wav", b"xx", "audio/wav")}).status_code
    print("transcribe:", t["text"], "| raw:", t["raw_text"])
    print("recommend/audio:", r["transcript"], "→ max_price", r["parsed"].get("max_price"), [p["id"] for p in r["products"]])
    print("speak:", s.status_code, s.headers.get("content-type"), "| bad audio →", bad)
    assert t["text"].endswith("$30") and r["parsed"].get("max_price") == 30 and s.status_code == 200 and bad == 400
    from src.ui.app import build_ui
    from src.ui.client import ShopTalkClient as UIClient
    build_ui(UIClient(fake.url), fake.url)
    print("fake-mode self-test ok")
finally:
    fake.stop()

## 3. Test set: your recordings + a synthetic set

### 📘 3.1 Choose the queries to record

**Purpose:** Pick `N_RECORD` labelled queries from the Phase 9 set, preferring the words speech-to-text gets wrong: brands, prices, sizes, numbers and colours (what the plan warns about), then fill up with others.

**Expected Results:** A table of the queries to record, saved to `record_list.csv`.

**Parameters:** `N_RECORD`.

In [ ]:
def asr_risk(q):
    pq = PARSER.parse(q)
    return (2 * bool(pq.brand) + 2 * (pq.max_price is not None or pq.min_price is not None) + 2 * bool(pq.size)
            + bool(re.search(r"\d", q)) + bool(pq.colors))
cands = [e for e in EVALSET if e["qrels"]]
ranked = sorted(cands, key=lambda e: (-asr_risk(e["query"]), e["qid"]))
RECORD_LIST = pd.DataFrame([{"qid": e["qid"], "query": e["query"], "risk": asr_risk(e["query"])} for e in ranked[:N_RECORD]])
RECORD_LIST.to_csv(RESULTS_DIR / "record_list.csv", index=False)
display(RECORD_LIST)

### 📘 3.2 🎤 Record the queries (interactive, 5–10 min)

**Purpose:** A small recorder page: it shows one query at a time; press record, read it naturally (as you would say it to a shopping assistant), stop, then **Save & next**. Recordings are stored on Drive as 16 kHz WAV under `recordings/<speaker>/<qid>.wav`, so you can stop and continue later. A friend can record too: they type their own speaker name. Open the **https** link printed below (the browser only allows the microphone on https).

**Expected Results:** Two links (a Gradio `gradio.live` link and a Cloudflare link) — open one in a normal browser tab; the page counter reaches 20/20. **Don't use “Run all” past this cell** — record first, then run 3.3. If your computer has no microphone, record on your phone and use the upload option on the same page.

**Parameters:** `SPEAKER`, `LAUNCH_RECORDER`, `REC_PORT`, `USE_CLOUDFLARED`.

In [ ]:
import gradio as gr
QUEUE = RECORD_LIST.to_dict("records")
def _spk(s): return re.sub(r"[^a-z0-9_-]", "", (s or "").strip().lower()) or "me"
def _path(spk, qid): return RECORD_DIR / _spk(spk) / f"{qid}.wav"
def _view(i, spk):
    q = QUEUE[i]; done = sum(_path(spk, x["qid"]).exists() for x in QUEUE)
    mark = "✅ already recorded — record again to replace" if _path(spk, q["qid"]).exists() else ""
    return f"#### {i + 1} / {len(QUEUE)} · speaker **{_spk(spk)}** · {done} saved\n## “{q['query']}”\n{mark}"
def _save(path, i, spk):
    if not path:
        return i, _view(i, spk) + "\n\n⚠️ Nothing recorded yet.", None
    try:
        x, sec = decode_audio(path, max_seconds=30)
    except AudioError as e:
        return i, _view(i, spk) + f"\n\n⚠️ {e}", None
    write_wav(_path(spk, QUEUE[i]["qid"]), x)
    j = min(i + 1, len(QUEUE) - 1)
    return j, _view(j, spk) + f"\n\nSaved {sec:.1f} s.", None
with gr.Blocks(title="ShopTalk recorder") as REC_APP:
    gr.Markdown("### ShopTalk — record test queries\nRead each request naturally. Any normal room is fine.")
    spk = gr.Textbox(value=SPEAKER, label="Speaker name (letters/numbers)")
    idx = gr.State(0)
    view = gr.Markdown(_view(0, SPEAKER))
    mic = gr.Audio(sources=["microphone", "upload"], type="filepath", format="wav",
                   label="Record (🎤) — or upload a file from your phone — then Save & next")
    with gr.Row():
        b_prev, b_skip, b_save = gr.Button("◀ Prev"), gr.Button("Skip ▶"), gr.Button("Save & next", variant="primary")
    b_save.click(_save, [mic, idx, spk], [idx, view, mic])
    b_skip.click(lambda i, s: (min(i + 1, len(QUEUE) - 1), _view(min(i + 1, len(QUEUE) - 1), s)), [idx, spk], [idx, view])
    b_prev.click(lambda i, s: (max(i - 1, 0), _view(max(i - 1, 0), s)), [idx, spk], [idx, view])
    spk.change(lambda i, s: _view(i, s), [idx, spk], view)
if LAUNCH_RECORDER:
    gr.close_all()                                   # stop an older recorder from a previous run of this cell
    subprocess.run(["pkill", "-f", f"127.0.0.1:{REC_PORT}"], check=False)   # and its old tunnel
    # inline=False: never show the recorder inside the notebook output (browsers block the microphone there)
    # share=True: Gradio's own https link (*.gradio.live); the Cloudflare tunnel is a second option
    _, _, share_url = REC_APP.launch(server_name="0.0.0.0", server_port=REC_PORT, prevent_thread_lock=True,
                                     inline=False, share=GRADIO_SHARE, quiet=True)
    alive = requests.get(f"http://127.0.0.1:{REC_PORT}/", timeout=10).status_code
    tunnel_url = start_tunnel(REC_PORT, "recorder_tunnel")
    print(f"recorder running locally: HTTP {alive}")
    print("🎤 Open ONE of these in a normal browser tab (not inside Colab):")
    print("   1.", share_url or "(Gradio share link unavailable)")
    print("   2.", tunnel_url or "(Cloudflare tunnel unavailable)")
    print("Keep this notebook open while you record; the page stops when the runtime stops.")

### 📘 3.3 Recording status

**Purpose:** Count the clips per speaker, play one back, and close the recorder.

**Expected Results:** `recordings_count` = PASS with ≥ `MIN_RECORDINGS` clips (plan: 10–20), else PENDING.

**Parameters:** `MIN_RECORDINGS`.

In [ ]:
CLIPS = []
for p in sorted(RECORD_DIR.glob("*/*.wav")):
    qid = p.stem
    if qid in QBY:
        x, sec = decode_audio(p)
        CLIPS.append({"set": "human", "speaker": p.parent.name, "qid": qid, "query": QBY[qid]["query"], "path": str(p), "audio_s": round(sec, 2)})
CLIPS_H = pd.DataFrame(CLIPS)
if len(CLIPS_H):
    display(CLIPS_H.groupby("speaker").agg(clips=("qid", "count"), mean_s=("audio_s", "mean")).round(1))
    display(Audio(CLIPS_H.iloc[0]["path"]))
check("recordings_count", "PASS" if len(CLIPS_H) >= MIN_RECORDINGS else "PENDING",
      f"{len(CLIPS_H)} recordings from {CLIPS_H['speaker'].nunique() if len(CLIPS_H) else 0} speaker(s) (need {MIN_RECORDINGS})")
if len(CLIPS_H) < MIN_RECORDINGS:
    print("⏸️ Not enough recordings yet: keep recording on the page from 3.2 (it is still running), then re-run this cell.")

### 📘 3.4 Synthetic set (computer voice, all queries)

**Purpose:** Read every eval query with the MMS voice (CPU) and save it to `synthetic/`. A clean computer voice gives an optimistic WER; comparing it with your recordings shows how much real speech costs. Reported separately.

**Expected Results:** 63 WAV files (≈ 1 min on CPU); `synthetic_set` = PASS.

**Parameters:** `MAKE_SYNTHETIC`.

In [ ]:
CLIPS_S = pd.DataFrame()
if MAKE_SYNTHETIC:
    from src.tts.mms_tts import MMSTTS
    tts_local = MMSTTS() if not USE_FAKE else None
    rows = []
    for e in EVALSET:
        p = SYN_DIR / f"{e['qid']}.wav"
        if not p.exists():
            if tts_local is None: continue
            p.parent.mkdir(parents=True, exist_ok=True); p.write_bytes(tts_local.synthesize(e["query"])["wav"])
        rows.append({"set": "synthetic", "speaker": "mms-tts", "qid": e["qid"], "query": e["query"], "path": str(p),
                     "audio_s": round(decode_audio(p)[1], 2)})
    CLIPS_S = pd.DataFrame(rows)
    del tts_local
    print(f"{len(CLIPS_S)} synthetic clips; example: “{CLIPS_S.iloc[0]['query']}”" if len(CLIPS_S) else "no synthetic clips")
    if len(CLIPS_S): display(Audio(CLIPS_S.iloc[0]["path"]))
check("synthetic_set", "PASS" if len(CLIPS_S) else "SKIP", f"{len(CLIPS_S)} synthetic clips")
ALL_CLIPS = pd.concat([CLIPS_H, CLIPS_S], ignore_index=True)

## 4. Compare Whisper model sizes (plan 10.1)

### 📘 4.1 Transcribe every clip with every candidate

**Purpose:** Load each Whisper model on the GPU (the API is not running yet, so it has the GPU to itself), warm it up, transcribe all clips and time each one; then free it. Results are cached in `asr_comparison_raw.csv`, so a re-run only adds new models or clips.

**Expected Results:** One row per model × clip with the raw transcript, the normalised query and the time. ≈ 3–8 min.

**Parameters:** `ASR_CANDIDATES`.

In [ ]:
import torch, gc
from src.asr.whisper_asr import WhisperASR
RAW_PATH = RESULTS_DIR / "asr_comparison_raw.csv"
done = pd.read_csv(RAW_PATH) if RAW_PATH.exists() else pd.DataFrame(columns=["model", "path"])
MODEL_INFO = json.loads((RESULTS_DIR / "asr_model_info.json").read_text()) if (RESULTS_DIR / "asr_model_info.json").exists() else {}
rows = done.to_dict("records")
for mid in ([] if USE_FAKE else ASR_CANDIDATES):
    todo = ALL_CLIPS[~ALL_CLIPS["path"].isin(done.loc[done["model"] == mid, "path"])]
    if todo.empty: continue
    print(f"→ {mid}: {len(todo)} clips")
    try:
        asr = WhisperASR(mid)
    except Exception as e:
        print("   skipped:", type(e).__name__, str(e)[:200]); continue
    MODEL_INFO[mid] = {"load_s": round(asr.load_s, 1), "gpu_mb": asr.gpu_mb()}
    asr.transcribe(decode_audio(todo.iloc[0]["path"])[0])          # warm-up (CUDA kernels, cache)
    for c in todo.itertuples():
        out = asr.transcribe(decode_audio(c.path)[0])
        rows.append({"model": mid, "set": c.set, "speaker": c.speaker, "qid": c.qid, "query": c.query, "path": c.path,
                     "audio_s": c.audio_s, "raw": out["text"], "hyp": transcript_to_query(out["text"]), "ms": out["ms"]})
    asr.unload(); del asr; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
ASR_RAW = pd.DataFrame(rows)
ASR_RAW.to_csv(RAW_PATH, index=False); (RESULTS_DIR / "asr_model_info.json").write_text(json.dumps(MODEL_INFO, indent=2))
print(len(ASR_RAW), "transcriptions")

### 📘 4.2 Word error rate vs speed — choose the default model

**Purpose:** Per model and set: corpus WER (raw transcript), key-word recall (brands, numbers, colours, product type), parsed-filter agreement with the typed query (raw and after number normalisation) and latency. Rule: among models whose P95 fits `ASR_P95_BUDGET_MS`, take the lowest WER on your recordings, then the fastest model within `ASR_WER_TOLERANCE` of it.

**Expected Results:** Comparison table, the chosen model, `asr_models_compared` and `asr_model_chosen` checks.

**Parameters:** `ASR_P95_BUDGET_MS`, `ASR_WER_TOLERANCE`, `ASR_MODEL`.

In [ ]:
def summarize_asr(df):
    out = []
    for (mid, st), g in df.groupby(["model", "set"]):
        fa_raw = [filter_agreement(r.query, r.raw, PARSER)["all"] for r in g.itertuples()]
        fa = [filter_agreement(r.query, r.hyp, PARSER)["all"] for r in g.itertuples()]
        ktr = [key_term_recall(r.query, r.raw, PARSER) for r in g.itertuples()]
        out.append({"model": mid, "set": st, "clips": len(g),
                    "WER": corpus_wer(zip(g["query"], g["raw"])),
                    "exact_match": float(np.mean([wer(r.query, r.raw) == 0 for r in g.itertuples()])),
                    "key_word_recall": float(np.nanmean([x for x in ktr if x is not None])) if any(x is not None for x in ktr) else np.nan,
                    "filters_match_raw": float(np.mean(fa_raw)), "filters_match": float(np.mean(fa)),
                    "p50_ms": float(np.percentile(g["ms"], 50)), "p95_ms": float(np.percentile(g["ms"], 95)),
                    "rtf": float((g["ms"] / 1000 / g["audio_s"]).median()),
                    "gpu_mb": MODEL_INFO.get(mid, {}).get("gpu_mb"), "load_s": MODEL_INFO.get(mid, {}).get("load_s")})
    return pd.DataFrame(out)

if len(ASR_RAW):
    ASR_CMP = summarize_asr(ASR_RAW)
    order = {m: i for i, m in enumerate(ASR_CANDIDATES)}
    ASR_CMP = ASR_CMP.sort_values(["set", "model"], key=lambda s: s.map(order) if s.name == "model" else s)
    display(ASR_CMP.round(3))
    ASR_CMP.to_csv(RESULTS_DIR / "asr_model_comparison.csv", index=False)
    basis = "human" if (ASR_CMP["set"] == "human").any() else "synthetic"
    b = ASR_CMP[ASR_CMP["set"] == basis].set_index("model")
    ok = b[b["p95_ms"] <= ASR_P95_BUDGET_MS]
    pool = ok if len(ok) else b
    best = pool["WER"].min()
    near = pool[pool["WER"] <= best + ASR_WER_TOLERANCE].sort_values("p95_ms")
    CHOSEN_ASR = ASR_MODEL or near.index[0]
    print(f"basis: {basis} clips · within budget: {list(ok.index)} · best WER {best:.3f} · chosen → {CHOSEN_ASR}")
    check("asr_models_compared", "PASS" if b.shape[0] >= 3 else "WARN", f"{b.shape[0]} models on {basis} clips")
    r = b.loc[CHOSEN_ASR]
    check("asr_model_chosen", "PASS" if r["p95_ms"] <= ASR_P95_BUDGET_MS else "WARN",
          f"{CHOSEN_ASR}: WER {r['WER']:.3f}, P95 {r['p95_ms']:.0f} ms ({basis})" + ("" if len(ok) else " — no model fits the budget"))
else:
    CHOSEN_ASR = ASR_MODEL or "openai/whisper-base.en"
    ASR_CMP = pd.DataFrame()
    check("asr_models_compared", "SKIP" if USE_FAKE else "FAIL", "no transcriptions (fake mode or no clips)")
    check("asr_model_chosen", "SKIP" if USE_FAKE else "WARN", CHOSEN_ASR)

### 📘 4.3 Where the models disagree

**Purpose:** Clips where at least one model got the words wrong, side by side — useful to see *which* words are misheard.

**Expected Results:** A table (reference vs each model's transcript), most errors first.

**Parameters:** none

In [ ]:
if len(ASR_RAW):
    ASR_RAW["wer"] = [wer(r.query, r.raw) for r in ASR_RAW.itertuples()]
    piv = ASR_RAW.pivot_table(index=["set", "speaker", "qid", "query"], columns="model", values="raw", aggfunc="first")
    bad = ASR_RAW[ASR_RAW["wer"] > 0].groupby(["set", "speaker", "qid"]).size().index
    show = piv.reset_index()
    show = show[show.set_index(["set", "speaker", "qid"]).index.isin(bad)]
    display(show.head(25))

### 📘 4.4 Save the choice in rag_config.json

**Purpose:** Write `asr: {model_id, max_new_tokens}` into the Drive `rag_config.json` (timestamped backup kept), so the API loads the chosen model at startup. `SHOPTALK_ASR_MODEL` still overrides it.

**Expected Results:** The new `asr` block printed.

**Parameters:** `APPLY_ASR_CHOICE`.

In [ ]:
if APPLY_ASR_CHOICE and not USE_FAKE:
    cfg_path = REPO_LINK / RAG_CONFIG_REL
    cfg = json.loads(cfg_path.read_text())
    if cfg.get("asr", {}).get("model_id") != CHOSEN_ASR:
        shutil.copy2(cfg_path, cfg_path.with_name(f"rag_config.json.bak-phase10-{RUN_ID}"))
        cfg["asr"] = {"model_id": CHOSEN_ASR, "max_new_tokens": 96, "chosen": f"phase10 {RUN_ID}"}
        cfg_path.write_text(json.dumps(cfg, indent=2))
    print(json.dumps(json.loads(cfg_path.read_text()).get("asr"), indent=2))
else:
    print("rag_config unchanged")

## 5. Start the API with speech-to-text

### 📘 5.1 Start the API and wait for `/health`

**Purpose:** Run uvicorn as a subprocess (repo on Drive, Chroma copied to local disk by the API, logs to `LOG_DIR`). Whisper is loaded once at startup, on the GPU next to Qwen; the read-aloud voice loads on CPU.

**Expected Results:** `/health` ok with `asr` = the chosen model and `asr_status` = ok (≈ 2–4 min).

**Parameters:** `API_APP`, `API_PORT`, `ENABLE_TTS`, `USE_FAKE`.

In [ ]:
from src.eval.api_server import ApiServer
from src.eval.client import ShopTalkClient as EvalClient
API_ENV = {"SHOPTALK_REPO_DIR": str(REPO_LINK), "SHOPTALK_LOCAL_CHROMA": LOCAL_CHROMA, "SHOPTALK_LOG_DIR": str(LOG_DIR),
           "SHOPTALK_ASR_MODEL": CHOSEN_ASR, "SHOPTALK_TTS": "1" if ENABLE_TTS else "0"}
if USE_FAKE: API_ENV["SHOPTALK_FAKE"] = "1"
else: os.environ.pop("SHOPTALK_FAKE", None)

def start_api():
    srv = ApiServer(REPO_LINK, API_APP, API_PORT, API_ENV, LOG_DIR / "uvicorn.log")
    srv.start()
    cl = EvalClient(srv.url)
    try:
        h = cl.wait_healthy(HEALTH_TIMEOUT_S, proc=srv.proc)
    except Exception:
        print(srv.tail(60)); raise
    return srv, cl, h

def ensure_api():
    global SERVER, CLIENT
    try:
        if CLIENT.health().get("status") == "ok": return
    except Exception: pass
    print("API not responding → restarting")
    try: SERVER.stop()
    except Exception: pass
    SERVER, CLIENT, _ = start_api()

SERVER, CLIENT, HEALTH = start_api()
check("api_health_voice", "PASS" if HEALTH.get("asr_status") == "ok" else "FAIL",
      f"asr={HEALTH.get('asr')} ({HEALTH.get('asr_status')}), tts={HEALTH.get('tts')} ({HEALTH.get('tts_status')})")

### 📘 5.2 Smoke test the voice endpoints

**Purpose:** One recording through `/transcribe` and `/recommend/audio`; check the response contract.

**Expected Results:** `transcribe_endpoint` and `recommend_audio_endpoint` = PASS.

**Parameters:** none

In [ ]:
def post_audio(endpoint, path, session_id):
    t0 = time.perf_counter()
    with open(path, "rb") as f:
        r = requests.post(SERVER.url + endpoint, files={"audio": (Path(path).name, f, "audio/wav")},
                          data={"session_id": session_id}, timeout=180)
    out = r.json() if r.headers.get("content-type", "").startswith("application/json") else {"detail": r.text[:300]}
    out["_client_ms"], out["_http_status"] = (time.perf_counter() - t0) * 1000, r.status_code
    return out
sample = (CLIPS_H if len(CLIPS_H) else ALL_CLIPS).iloc[0]
t = post_audio("/transcribe", sample["path"], f"{SESSION_PREFIX}-smoke")
r = post_audio("/recommend/audio", sample["path"], f"{SESSION_PREFIX}-smoke")
print("said:", sample["query"]); print("heard:", t.get("raw_text"), "→ query:", t.get("text"), t.get("latency_ms"))
print("answer:", (r.get("answer") or "")[:200])
check("transcribe_endpoint", "PASS" if t["_http_status"] == 200 and {"text", "raw_text", "latency_ms"} <= set(t) else "FAIL",
      f"http {t['_http_status']}, asr {t.get('latency_ms', {}).get('asr')} ms")
check("recommend_audio_endpoint", "PASS" if r["_http_status"] == 200 and {"transcript", "products", "latency_ms"} <= set(r) else "FAIL",
      f"http {r['_http_status']}, total_with_asr {r.get('latency_ms', {}).get('total_with_asr')} ms")

## 6. Voice vs typed queries through the API (plan 10.5)

### 📘 6.1 Run every clip and its typed twin

**Purpose:** Each recording (and, if enabled, each synthetic clip) goes through `/recommend/audio`; the typed reference query goes through `/recommend`. Same pipeline, so any difference comes from speech-to-text.

**Expected Results:** `voice_eval_runs.jsonl`; ≈ 4–5 s per request.

**Parameters:** `RUN_SYNTH_E2E`.

In [ ]:
from tqdm.auto import tqdm
ensure_api()
clips = pd.concat([CLIPS_H, CLIPS_S if RUN_SYNTH_E2E else CLIPS_S.iloc[0:0]], ignore_index=True)
TEXT_RUNS = {}
for q in tqdm(sorted(set(clips["query"])), desc="typed"):
    TEXT_RUNS[q] = CLIENT.recommend(q, f"{SESSION_PREFIX}-text")
VOICE_RUNS = []
for c in tqdm(clips.itertuples(), total=len(clips), desc="voice"):
    v = post_audio("/recommend/audio", c.path, f"{SESSION_PREFIX}-voice-{c.speaker}")
    # the response's own "query" is what was searched (the transcript); keep the spoken reference as "query"
    VOICE_RUNS.append({**v, "searched": v.get("query"), "set": c.set, "speaker": c.speaker, "qid": c.qid,
                       "query": c.query, "audio_s": c.audio_s})
with open(RESULTS_DIR / "voice_eval_runs.jsonl", "w") as f:
    for r in VOICE_RUNS: f.write(json.dumps(r, default=str) + "\n")
errs = sum(r["_http_status"] != 200 for r in VOICE_RUNS)
print(f"{len(VOICE_RUNS)} voice requests, {errs} errors; {len(TEXT_RUNS)} typed requests")

### 📘 6.2 WER and retrieval precision, measured separately

**Purpose:** Per set: corpus WER and key-word recall (speech-to-text quality); parsed-filter agreement with the typed query (does mishearing change the search?); and P@5 for voice vs typed on the same queries, plus how many of the top-5 products are the same.

**Expected Results:** Summary table; checks `wer_human`, `filter_agreement_human`, `voice_vs_text_precision`.

**Parameters:** `WER_TARGET`, `FILTER_AGREEMENT_TARGET`, `PRECISION_DROP_TOLERANCE`.

In [ ]:
rows = []
for r in VOICE_RUNS:
    e, t = QBY[r["qid"]], TEXT_RUNS[r["query"]]
    vids = [str(p["id"]) for p in r.get("products", [])]
    tids = [str(p["id"]) for p in t.get("products", [])]
    fa = filter_agreement(r["query"], r.get("transcript", ""), PARSER)
    rows.append({"set": r["set"], "speaker": r["speaker"], "qid": r["qid"], "query": r["query"],
                 "raw_transcript": r.get("raw_transcript", ""), "transcript": r.get("transcript", ""),
                 "wer": wer(r["query"], r.get("raw_transcript", "")),
                 "key_word_recall": key_term_recall(r["query"], r.get("raw_transcript", ""), PARSER),
                 "filters_match": fa["all"], "filters_mismatched": ",".join(fa["mismatched"]),
                 "p5_voice": M.precision_at_k(vids, e["qrels"], 5) if e["qrels"] else np.nan,
                 "p5_text": M.precision_at_k(tids, e["qrels"], 5) if e["qrels"] else np.nan,
                 "same_products@5": len(set(vids[:5]) & set(tids[:5])) / max(len(set(tids[:5])), 1),
                 "no_match_voice": r.get("no_match"), "no_match_text": t.get("no_match"),
                 "http": r["_http_status"]})
VOICE_EVAL = pd.DataFrame(rows)
VOICE_EVAL.to_csv(RESULTS_DIR / "voice_eval_per_clip.csv", index=False)
def set_summary(g):
    return pd.Series({"clips": len(g), "WER": corpus_wer(zip(g["query"], g["raw_transcript"])),
                      "exact_transcripts": (g["wer"] == 0).mean(),
                      "key_word_recall": g["key_word_recall"].dropna().mean(), "filters_match": g["filters_match"].mean(),
                      "P@5_voice": g["p5_voice"].mean(), "P@5_typed": g["p5_text"].mean(),
                      "ΔP@5": (g["p5_voice"] - g["p5_text"]).mean(), "same_products@5": g["same_products@5"].mean()})
VOICE_SUM = VOICE_EVAL.groupby("set").apply(set_summary)
VOICE_SUM["clips"] = VOICE_SUM["clips"].astype(int)
display(VOICE_SUM.round(3))
VOICE_SUM.to_csv(RESULTS_DIR / "voice_eval_summary.csv")
if "human" in VOICE_SUM.index:
    h = VOICE_SUM.loc["human"]
    check("wer_human", "PASS" if h["WER"] <= WER_TARGET else "WARN", f"corpus WER {h['WER']:.3f} on {int(h['clips'])} recordings (target ≤ {WER_TARGET})")
    check("filter_agreement_human", "PASS" if h["filters_match"] >= FILTER_AGREEMENT_TARGET else "WARN",
          f"{h['filters_match']:.0%} of recordings parse to the same filters as the typed query")
    d = h["ΔP@5"]
    check("voice_vs_text_precision", "PASS" if d >= -PRECISION_DROP_TOLERANCE else "WARN",
          f"P@5 voice {h['P@5_voice']:.3f} vs typed {h['P@5_typed']:.3f} (Δ {d:+.3f}); same top-5 products {h['same_products@5']:.0%}")
else:
    for k in ("wer_human", "filter_agreement_human", "voice_vs_text_precision"):
        check(k, "PENDING", "no recordings yet (3.2)")

### 📘 6.3 Misrecognitions that changed the results

**Purpose:** The clips where the transcript differs from what was said, with the filters that changed and the effect on P@5 / shared products. These are the voice failure cases for the write-up.

**Expected Results:** A table sorted by impact; saved as `voice_failures.csv`.

**Parameters:** none

In [ ]:
VOICE_FAIL = VOICE_EVAL[(VOICE_EVAL["wer"] > 0)].assign(
    impact=lambda d: (~d["filters_match"]).astype(int) * 2 + (d["same_products@5"] < 1).astype(int)
                     + (d["p5_voice"] < d["p5_text"]).fillna(False).astype(int))
VOICE_FAIL = VOICE_FAIL.sort_values(["impact", "wer"], ascending=False)
cols = ["set", "speaker", "qid", "query", "raw_transcript", "transcript", "wer", "filters_mismatched", "p5_voice", "p5_text", "same_products@5"]
display(VOICE_FAIL[cols].head(20))
VOICE_FAIL[cols].to_csv(RESULTS_DIR / "voice_failures.csv", index=False)
fixed = VOICE_EVAL.apply(lambda r: filter_agreement(r["query"], r["transcript"], PARSER)["all"]
                         and not filter_agreement(r["query"], r["raw_transcript"], PARSER)["all"], axis=1).sum()
print(f"number normalisation (raw → query) fixed the parsed filters on {fixed} clip(s)")

## 7. Latency with speech-to-text (plan 10.5)

### 📘 7.1 Speech-to-text latency: 100 `/transcribe` calls

**Purpose:** Cycle through the clips (recordings first) and measure decode, GPU queue and model time.

**Expected Results:** `asr_latency_raw.csv`; ≈ 1 min.

**Parameters:** `N_ASR_LATENCY`, `LATENCY_WARMUP`.

In [ ]:
import src.eval.latency as LT
ensure_api()
paths = list(CLIPS_H["path"]) + list(CLIPS_S["path"]) if len(CLIPS_H) else list(ALL_CLIPS["path"])
def asr_call(p, i):
    t = post_audio("/transcribe", p, f"{SESSION_PREFIX}-lat-asr")
    lat = t.get("latency_ms") or {}
    return {"http": t["_http_status"], "client_ms": t["_client_ms"], **{f"lat_{k}": v for k, v in lat.items()}, "audio_s": t.get("audio_s")}
ASR_LAT = pd.DataFrame(LT.run_requests(asr_call, paths, N_ASR_LATENCY, LATENCY_WARMUP, desc="transcribe"))
ASR_LAT.to_csv(RESULTS_DIR / "asr_latency_raw.csv", index=False)
ASR_LAT_OK = ASR_LAT[ASR_LAT["ok"] & (ASR_LAT["http"] == 200)]
ASR_TABLE = LT.latency_table(ASR_LAT_OK, {"decode": "lat_decode", "queue": "lat_queue", "asr (model)": "lat_asr",
                                          "/transcribe round-trip": "client_ms"})
display(ASR_TABLE)
p95_asr = ASR_TABLE.set_index("stage").loc["asr (model)", "p95"]
check("asr_latency_p95", "PASS" if p95_asr <= ASR_P95_BUDGET_MS else "WARN", f"speech-to-text P95 {p95_asr:,.0f} ms on {len(ASR_LAT_OK)} calls (budget {ASR_P95_BUDGET_MS} ms)")

### 📘 7.2 Voice end to end: 100 `/recommend/audio` calls

**Purpose:** Full voice requests (speech-to-text + parse + retrieve/rerank + generate), sequential like Phase 9.

**Expected Results:** `voice_latency_raw.csv`; ≈ 8 min.

**Parameters:** `N_VOICE_E2E`.

In [ ]:
ensure_api()
def voice_call(p, i):
    v = post_audio("/recommend/audio", p, f"{SESSION_PREFIX}-lat-voice")
    return {"http": v["_http_status"], "client_ms": v["_client_ms"], **{f"lat_{k}": x for k, x in (v.get("latency_ms") or {}).items()}}
VOICE_LAT = pd.DataFrame(LT.run_requests(voice_call, paths, N_VOICE_E2E, LATENCY_WARMUP, desc="recommend/audio"))
VOICE_LAT.to_csv(RESULTS_DIR / "voice_latency_raw.csv", index=False)
VOICE_LAT_OK = VOICE_LAT[VOICE_LAT["ok"] & (VOICE_LAT["http"] == 200)]
check("voice_latency_n", "PASS" if len(VOICE_LAT_OK) >= 100 else "WARN", f"{len(VOICE_LAT_OK)} successful voice requests")

### 📘 7.3 The combined P50/P95/P99 report

**Purpose:** Phase 9's text stages plus speech-to-text: one table for the write-up, with both paths side by side.

**Expected Results:** `latency_report.csv` / `.md`; `voice_e2e_p95` check.

**Parameters:** `VOICE_E2E_TARGET_MS`.

In [ ]:
VOICE_TABLE = LT.latency_table(VOICE_LAT_OK, {"decode": "lat_decode", "asr": "lat_asr", "parse": "lat_parse",
                                              "retrieve (+rerank)": "lat_retrieve", "generate": "lat_generate",
                                              "queue": "lat_queue", "server total (text part)": "lat_total",
                                              "server total incl. speech": "lat_total_with_asr", "client round-trip": "client_ms"})
VOICE_TABLE.insert(0, "path", "voice (Phase 10)")
p9 = PHASE9_DIR / "latency_summary.csv"
TEXT_TABLE = pd.read_csv(p9).assign(path="text (Phase 9)") if p9.exists() else pd.DataFrame()
LAT_REPORT = pd.concat([TEXT_TABLE[["path"] + [c for c in TEXT_TABLE.columns if c != "path"]] if len(TEXT_TABLE) else TEXT_TABLE,
                        VOICE_TABLE], ignore_index=True)
display(LAT_REPORT)
LAT_REPORT.to_csv(RESULTS_DIR / "latency_report.csv", index=False)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vt = VOICE_TABLE.set_index("stage")
p95_voice = vt.loc["client round-trip", "p95"]
share = (pd.to_numeric(VOICE_LAT_OK["lat_asr"]) / pd.to_numeric(VOICE_LAT_OK["lat_total_with_asr"])).median()
print(f"speech-to-text is {share:.1%} of a voice request (median) on {gpu}")
check("voice_e2e_p95", "PASS" if p95_voice <= VOICE_E2E_TARGET_MS else "WARN",
      f"voice client P95 {p95_voice:,.0f} ms on {gpu} (target ≤ {VOICE_E2E_TARGET_MS:,})")
check("latency_report", "PASS" if len(TEXT_TABLE) else "WARN",
      "text + voice percentiles in latency_report.csv" if len(TEXT_TABLE) else "Phase 9 latency_summary.csv not found — voice only")

## 8. Optional: text-to-speech (plan 10.4)

### 📘 8.1 Read-aloud check

**Purpose:** Synthesize 10 answers from §6 with `/speak`, time it, and play one. The UI checkbox stays off by default.

**Expected Results:** TTS P50/P95 and an audio player; `tts_endpoint` = PASS (or SKIP when `ENABLE_TTS=False`).

**Parameters:** `ENABLE_TTS`.

In [ ]:
if ENABLE_TTS:
    answers = [r.get("answer", "") for r in VOICE_RUNS if r.get("answer")][:10]
    tts_rows, last = [], None
    for a in answers:
        t0 = time.perf_counter(); s = requests.post(SERVER.url + "/speak", json={"text": a}, timeout=120)
        tts_rows.append({"http": s.status_code, "client_ms": (time.perf_counter() - t0) * 1000,
                         "tts_ms": float(s.headers.get("X-TTS-ms", "nan")), "audio_s": float(s.headers.get("X-Audio-Seconds", "nan"))})
        if s.status_code == 200: last = s.content
    TTS = pd.DataFrame(tts_rows)
    display(LT.latency_table(TTS, {"tts (model)": "tts_ms", "/speak round-trip": "client_ms", "audio length (s)": "audio_s"}))
    if last: (RESULTS_DIR / "tts_sample.wav").write_bytes(last); display(Audio(str(RESULTS_DIR / "tts_sample.wav")))
    check("tts_endpoint", "PASS" if len(TTS) and (TTS["http"] == 200).all() else "WARN", f"{(TTS['http'] == 200).sum()}/{len(TTS)} answers spoken")
else:
    check("tts_endpoint", "SKIP", "ENABLE_TTS=False")

## 9. UI with microphone (plan 10.3)

### 📘 9.1 Launch the UI

**Purpose:** Start the Phase 8 UI (now with 🎤 and 🔊) as a subprocess and open it through an https tunnel.

**Expected Results:** A `https://…trycloudflare.com` link.

**Parameters:** `UI_PORT`, `USE_CLOUDFLARED`.

In [ ]:
UI_PROC = subprocess.Popen([sys.executable, "-m", "src.ui.app"], cwd=REPO_LINK,
                           env={**os.environ, "PYTHONPATH": str(REPO_LINK), "SHOPTALK_API_URL": SERVER.url, "SHOPTALK_UI_PORT": str(UI_PORT)},
                           stdout=open(LOG_DIR / "ui.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{UI_PORT}/ui-health", timeout=2).ok: break
    except Exception: time.sleep(1)
UI_URL = start_tunnel(UI_PORT, "ui_tunnel")
print("🛍️ ShopTalk UI:", UI_URL or f"http://127.0.0.1:{UI_PORT} (no tunnel)")

### 📘 9.2 Manual test (you)

**Purpose:** In the UI: (1) press 🎤, say a request with a brand or a price, stop — the words appear in the text box and under it (*Heard: …*); (2) fix a word if needed and press **Send**; (3) tick 🔊 and send another request — the answer is read aloud; (4) say something unclear or very short — you get a friendly message, not an error. Then set `UI_VOICE_TESTED = True` below (and add notes).

**Expected Results:** `ui_voice_manual` = PASS after you set the flag.

**Parameters:** `UI_VOICE_TESTED`, `UI_NOTES`.

In [ ]:
UI_VOICE_TESTED = False
UI_NOTES = ""        # e.g. "transcript appeared in ~0.5 s; 'Symbol' heard as 'cymbal' once"
check("ui_voice_manual", "PASS" if UI_VOICE_TESTED else "PENDING", UI_NOTES or "set UI_VOICE_TESTED = True after testing 9.2")

## 10. Write-up

### 📘 10.1 `asr_report.md`

**Purpose:** Model comparison, the chosen model, voice-vs-typed accuracy, misrecognitions that changed results, the combined latency table and a short list of next steps (to edit).

**Expected Results:** `results/phase10/asr_report.md`, displayed below; `asr_writeup` = PASS.

**Parameters:** none

In [ ]:
def md_table(df):
    if df is None or len(df) == 0: return "_(none)_\n"
    d = df.copy()
    for c in d.columns:
        if d[c].dtype.kind == "f": d[c] = d[c].map(lambda v: "" if pd.isna(v) else (f"{v:.3f}" if abs(v) < 100 else f"{v:,.0f}"))
    return "| " + " | ".join(map(str, d.columns)) + " |\n|" + "---|" * len(d.columns) + "\n" + "\n".join(
        "| " + " | ".join(str(v).replace("|", "/").replace("\n", " ") for v in r) + " |" for r in d.itertuples(index=False)) + "\n"
h = VOICE_SUM.loc["human"] if "human" in VOICE_SUM.index else None
lines = [f"# ShopTalk Phase 10 — Voice input", f"_Run {RUN_ID} on {gpu}_\n",
         "## 1. Summary\n",
         f"- Speech-to-text model: **{CHOSEN_ASR}**, loaded once at API startup.",
         f"- Recordings: {len(CLIPS_H)} clips from {CLIPS_H['speaker'].nunique() if len(CLIPS_H) else 0} speaker(s); synthetic: {len(CLIPS_S)} clips.",
         (f"- On your recordings: WER **{h['WER']:.3f}**, same parsed filters as typed **{h['filters_match']:.0%}**, "
          f"P@5 voice **{h['P@5_voice']:.3f}** vs typed **{h['P@5_typed']:.3f}**." if h is not None else "- No recordings yet."),
         f"- Speech-to-text P95 **{p95_asr:,.0f} ms**; voice request P95 **{p95_voice:,.0f} ms** (client).",
         "\n## 2. Whisper model comparison\n", md_table(ASR_CMP.round(3) if len(ASR_CMP) else ASR_CMP),
         "\n## 3. Voice vs typed (same pipeline)\n", md_table(VOICE_SUM.reset_index().round(3)),
         "\n## 4. Misrecognitions that mattered\n", md_table(VOICE_FAIL[cols].head(15)),
         "\n## 5. Latency (P50 / P95 / P99, ms)\n", md_table(LAT_REPORT),
         "\n## 6. Findings and next steps (edit)\n",
         "- Spoken numbers: the rule parser only reads digits, and the type lexicon maps \"thirty\" to HAT, so transcripts "
         "are normalised (\"under thirty dollars\" → \"under $30\") before searching. Section 6.3 counts how many clips this fixed.",
         "- Brands and product names are the usual misrecognitions; candidate fixes: Whisper `prompt_ids` with catalog brand "
         "names, or fuzzy-matching transcript words against `brands.json`.",
         "- The UI never searches a transcript blindly: the user sees and can edit it first.",
         "- Production (Phase 11): the API image needs `ffmpeg` and `python-multipart`; Whisper adds its GPU memory to Qwen's."]
REPORT = "\n".join(lines)
(RESULTS_DIR / "asr_report.md").write_text(REPORT)
display(Markdown(REPORT))
check("asr_writeup", "PASS", "asr_report.md written")

## 11. Exit checklist

### 📘 11.1 Stop services, archive logs, save the checklist

**Purpose:** Stop the API, UI and tunnels; copy the Phase 10 logs to `results/phase10/logs/`; save `summary.json` and `exit_checklist.json/.md`.

**Expected Results:** Phase 10 is **done** when there is no FAIL and no PENDING.

**Parameters:** none

In [ ]:
try: REC_APP.close()
except Exception: pass
for name in ("UI_PROC",):
    p = globals().get(name)
    if p and p.poll() is None:
        os.killpg(p.pid, 15)
try: SERVER.stop()
except Exception as e: print("stop:", e)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
arch = RESULTS_DIR / "logs"; arch.mkdir(exist_ok=True)
for f in LOG_DIR.glob("*.*"):
    if f.is_file(): shutil.copy2(f, arch / f.name)
save_json({"run_id": RUN_ID, "gpu": gpu, "asr_model": CHOSEN_ASR, "recordings": len(CLIPS_H), "synthetic": len(CLIPS_S),
           "asr_comparison": ASR_CMP.to_dict("records") if len(ASR_CMP) else None,
           "voice_vs_text": VOICE_SUM.reset_index().to_dict("records"), "latency": LAT_REPORT.to_dict("records")}, "summary.json")
order = ["recordings_count", "synthetic_set", "asr_models_compared", "asr_model_chosen", "api_health_voice",
         "transcribe_endpoint", "recommend_audio_endpoint", "wer_human", "filter_agreement_human", "voice_vs_text_precision",
         "asr_latency_p95", "voice_latency_n", "voice_e2e_p95", "latency_report", "tts_endpoint", "ui_voice_manual", "asr_writeup"]
EXIT = pd.DataFrame([{"check": k, **CHECKS.get(k, {"status": "MISSING", "detail": "cell not run"})} for k in order])
EXIT["status"] = EXIT["status"].map(lambda s: f"{_ICON.get(s, '❓')} {s}")
display(EXIT)
save_json({"run_id": RUN_ID, "checks": CHECKS}, "exit_checklist.json")
(RESULTS_DIR / "exit_checklist.md").write_text("# Phase 10 exit checklist\n\n| check | status | detail |\n|---|---|---|\n" +
    "\n".join(f"| {r.check} | {r.status} | {str(r.detail).replace('|', '/')} |" for r in EXIT.itertuples()))
st = [CHECKS.get(k, {}).get("status", "MISSING") for k in order]
print(f"\nPASS {st.count('PASS')} · WARN {st.count('WARN')} · FAIL {st.count('FAIL')} · PENDING {st.count('PENDING')} · SKIP {st.count('SKIP')} · MISSING {st.count('MISSING')}")
print("✅ Phase 10 complete" if not ({"FAIL", "PENDING", "MISSING"} & set(st)) else "⏳ Not done yet: resolve FAIL / PENDING / MISSING above")
print("Results:", RESULTS_DIR)